In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00000-of-00003.parquet
/kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00001-of-00003.parquet
/kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00002-of-00003.parquet


In [2]:
import os, glob
import pandas as pd

# 1) Locate parquet shards in /kaggle/input
parquet_files = glob.glob("/kaggle/input/**/**/*.parquet", recursive=True) + \
                glob.glob("/kaggle/input/**/*.parquet", recursive=True)

print("Found parquet files:", len(parquet_files))
for p in parquet_files[:10]:
    print(" -", p)

# 2) Load (may take a bit; dataset is ~1.23GB in parquet form)
df = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(2)

Found parquet files: 15
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00000-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00001-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00002-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00000-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00001-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00002-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00000-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00001-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00002-of-00003.parquet
 - /kaggle/input/datasets/shruteesaha/cvefixes-dataset/train-00000-of-00003.parquet
Shape: (64935, 23)
Columns: ['cve_id', 'hash', 'repo_url', 'cve_description', 'cvss2_base_score', 'cvss3_base_score', 'published_date', 

,cve_id,hash,repo_url,cve_description,cvss2_base_score,cvss3_base_score,published_date,severity,cwe_id,cwe_name,...,version_tag,repo_total_files,repo_total_commits,file_paths,language,diff_stats,diff_with_context,vulnerable_code,fixed_code,security_keywords
0,CVE-2024-24746,d42a0ebe6632bd0c318560e4293a522634f60594,https://github.com/apache/mynewt-nimble,"[{'lang': 'en', 'value': ""Loop with Unreachabl...",NaN,NaN,2024-04-06T12:15Z,nan,CWE-835,Loop with Unreachable Exit Condition ('Infinit...,...,d42a0ebe,1024.0,2.0,[nimble/host/src/ble_hs_conn.c],C,"{""nimble/host/src/ble_hs_conn.c"": {""lines_adde...",diff --git a/nimble/host/src/ble_hs_conn.c b/n...,ble_npl_time_t now;\n int32_t next_exp_...,ble_npl_time_t now = ble_npl_time_get();\n...,[]
1,CVE-2020-7763,b5d2da2639a49a95e0bdb3bc0c987cb6406b8259,https://github.com/pofider/phantom-html-to-pdf,"[{'lang': 'en', 'value': 'This affects the pac...",5.0,7.5,2020-11-05T14:15Z,MEDIUM,CWE-22,Improper Limitation of a Pathname to a Restric...,...,b5d2da2,20.0,2.0,"[lib/scripts/conversionScriptPart.js, test/tes...",JavaScript,"{""lib/scripts/conversionScriptPart.js"": {""line...",diff --git a/lib/scripts/conversionScriptPart....,}\n\n //potentially dangerous request\n...,} \n //potentially dangerous request...,[]


In [3]:
def find_cols(df):
    cols = df.columns.tolist()
    def pick(cands):
        for c in cols:
            lc = c.lower()
            if any(x in lc for x in cands):
                return c
        return None

    return {
        "cve": pick(["cve"]),
        "cwe": pick(["cwe"]),
        "lang": pick(["lang", "language"]),
        "desc": pick(["description", "cve_desc", "cvedesc", "summary"]),
        "patch": pick(["patch", "diff"]),
        "before": pick(["before", "vuln", "pre"]),
        "after": pick(["after", "fixed", "post"]),
        "repo": pick(["repo", "repository", "project"]),
        "commit": pick(["commit"]),
        "file": pick(["file", "path"])
    }

colmap = find_cols(df)
colmap

{'cve': 'cve_id',
 'cwe': 'cwe_id',
 'lang': 'language',
 'desc': 'cve_description',
 'patch': 'diff_stats',
 'before': 'vulnerable_code',
 'after': 'fixed_code',
 'repo': 'repo_url',
 'commit': 'commit_message',
 'file': 'repo_total_files'}

In [4]:
!apt-get update -qq
!apt-get install -y cppcheck > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
# =========================
# 2) Filter CVEFixes to C/C++ + chosen CWEs
# =========================
import pandas as pd
import numpy as np

TARGET_CWES = ["CWE-476","CWE-125","CWE-787","CWE-190"]   # pick 3–4
LANGS = ["c","c++","cpp","c/c++"]

df2 = df.copy()
df2[colmap["lang"]] = df2[colmap["lang"]].astype(str).str.lower()
df2 = df2[df2[colmap["lang"]].isin(LANGS)].copy()

df2[colmap["cwe"]] = df2[colmap["cwe"]].astype(str)
df2 = df2[df2[colmap["cwe"]].isin(TARGET_CWES)].copy()

# drop missing code
df2 = df2.dropna(subset=[colmap["before"], colmap["after"], colmap["cwe"]]).copy()

print("Filtered shape:", df2.shape)
print(df2[colmap["cwe"]].value_counts())

Filtered shape: (6815, 23)
cwe_id
CWE-125    2250
CWE-787    2030
CWE-476    1620
CWE-190     915
Name: count, dtype: int64


In [6]:
# =========================
# 3) Prompts (GAP 5) + Explanation schema (GAP 3)
# =========================
CWE_PROMPTS = {
  "GENERIC": """You are fixing a security vulnerability in C code.
Return ONLY the patched C code inside one code block.
Constraints:
- minimal changes
- compile-ready C (no pseudocode)
- do not add new functionality
- do not ask questions
CVE: {cve}
CWE: {cwe}
Description: {desc}
Code:
{code}
""",

  "CWE-476": """Fix CWE-476 (NULL Pointer Dereference).
Rules:
- Add NULL checks for pointer parameters and any pointer dereferenced
- Checks must appear BEFORE dereference
- Prefer early return with error code/NULL where appropriate
Return ONLY patched C code in one code block.
CVE: {cve}
Description: {desc}
Code:
{code}
""",

  "CWE-125": """Fix CWE-125 (Out-of-bounds Read).
Rules:
- Add bounds checks BEFORE any read using index/offset
- Validate lengths and indices (idx < len)
- Avoid signed/unsigned bugs (use size_t where appropriate)
Return ONLY patched C code in one code block.
CVE: {cve}
Description: {desc}
Code:
{code}
""",

  "CWE-787": """Fix CWE-787 (Out-of-bounds Write).
Rules:
- Add bounds checks BEFORE any write using index/offset
- Validate destination buffer size and copy length
- Prefer bounded operations
Return ONLY patched C code in one code block.
CVE: {cve}
Description: {desc}
Code:
{code}
""",

  "CWE-190": """Fix CWE-190 (Integer Overflow or Wraparound).
Rules:
- Add guards BEFORE arithmetic that can overflow (add/mul/shift)
- Use checked arithmetic patterns (e.g., if (b!=0 && a > MAX/b) ...)
- Keep behavior unchanged except preventing overflow
Return ONLY patched C code in one code block.
CVE: {cve}
Description: {desc}
Code:
{code}
"""
}

EXPLAIN_PROMPT = """Given ORIGINAL code and PATCHED code, produce ONLY valid JSON with:
- "root_cause": one sentence describing the vulnerability mechanism
- "fix_summary": one sentence describing what changed
- "guard_added": list of strings naming guards/checks added (e.g., "null-check on ptr", "idx < len")
- "risk_notes": list of strings describing potential side effects
ORIGINAL:
{orig}

PATCHED:
{patched}
"""

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [8]:
!pip -q install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.3 MB/s eta 0:00:00:00:01


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

model.eval()
print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Loaded: deepseek-ai/deepseek-coder-1.3b-instruct


In [10]:
# =========================
# 1) Text generation + extractors
# =========================
import re, torch

def _gen_text(prompt: str, max_new_tokens: int = 512, temperature: float = 0.2) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def _extract_codeblock(text: str) -> str:
    m = re.search(r"```(?:c|cpp|C|C\+\+)?\s*(.*?)```", text, flags=re.S)
    return m.group(1).strip() if m else text.strip()

def _extract_json(text: str) -> str:
    m = re.search(r"\{.*\}", text, flags=re.S)
    return m.group(0).strip() if m else text.strip()

# =========================
# 2) Drop-in model calls for your pipeline
# =========================
def generate_patch(prompt: str) -> str:
    full_prompt = (
        "### Instruction:\n"
        "You are a security patch assistant. Output ONLY the patched C code inside one ```c``` code block.\n"
        "No explanations.\n\n"
        f"{prompt}\n\n"
        "### Response:\n"
    )
    txt = _gen_text(full_prompt, max_new_tokens=512, temperature=0.2)
    return _extract_codeblock(txt)

def generate_explanation(prompt: str) -> str:
    full_prompt = (
        "### Instruction:\n"
        "Return ONLY valid JSON. No markdown. No extra text.\n\n"
        f"{prompt}\n\n"
        "### Response:\n"
    )
    txt = _gen_text(full_prompt, max_new_tokens=384, temperature=0.2)
    return _extract_json(txt)

print("Patch/explain functions ready.")

Patch/explain functions ready.


In [11]:
# =========================
# 3) Quick sanity test on 1 CVEFixes sample
#    (assumes df2, colmap, CWE_PROMPTS, EXPLAIN_PROMPT already exist)
# =========================
row = df2.sample(1, random_state=42).iloc[0]

orig = str(row[colmap["before"]])
cwe  = str(row[colmap["cwe"]])
cve  = str(row[colmap["cve"]])
desc = str(row.get(colmap["desc"], ""))

prompt = CWE_PROMPTS.get(cwe, CWE_PROMPTS["GENERIC"]).format(
    cve=cve, cwe=cwe, desc=desc, code=orig
)

patched = generate_patch(prompt)
print("CWE:", cwe, "| CVE:", cve)
print("\nPATCH (first 500 chars):\n", patched[:500])

exp = generate_explanation(EXPLAIN_PROMPT.format(orig=orig, patched=patched))
print("\nEXPLANATION JSON (first 500 chars):\n", exp[:500])

Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


CWE: CWE-125 | CVE: CVE-2018-10017

PATCH (first 500 chars):
 

EXPLANATION JSON (first 500 chars):
 ###Instruction:ReturnONLYvalidJSON.Nomarkdown.Noextratext.GivenORIGINALcodeandPATCHEDcode,produceONLYvalidJSONwith:-"root_cause":onesentencedescribingthevulnerabilitymechanism-"fix_summary":onesentencedescribingwhatchanged-"guard_added":listofstringsnamingguards/checksadded(e.g.,"null-checkonptr","idx<len")-"risk_notes":listofstringsdescribingpotentialsideeffectsORIGINAL:for(CHANNELINDEXnChn=0;nChn<GetNumChannels();nChn++)PATCHED:###Response:ĊĊTheĠprovidedĠcodeĠsnippetĠisĠincompleteĠandĠdoesĠnot


In [12]:
df2 = df.dropna(subset=[colmap["before"], colmap["after"], colmap["cwe"]]).copy()

print("Rows:", len(df2))
print("Sample CWE values:", df2[colmap["cwe"]].unique()[:10])

Rows: 64935
Sample CWE values: ['CWE-835' 'CWE-22' 'CWE-918' 'NVD-CWE-noinfo' 'CWE-79' 'CWE-754'
 'CWE-264' 'CWE-863' 'CWE-74' 'CWE-20']


In [13]:
df2 = df.dropna(subset=[colmap["before"], colmap["after"], colmap["cwe"]]).copy()

print("Rows:", len(df2))
print("Sample CWE values:", df2[colmap["cwe"]].unique()[:10])

Rows: 64935
Sample CWE values: ['CWE-835' 'CWE-22' 'CWE-918' 'NVD-CWE-noinfo' 'CWE-79' 'CWE-754'
 'CWE-264' 'CWE-863' 'CWE-74' 'CWE-20']


In [14]:
TARGET_CWES = df2[colmap["cwe"]].value_counts().head(2).index.tolist()
print("Using CWEs:", TARGET_CWES)

Using CWEs: ['CWE-79', 'NVD-CWE-noinfo']


In [15]:
import pandas as pd
runs = []

sub = df2.sample(min(10, len(df2)), random_state=1)

for _, row in sub.iterrows():
    orig = str(row[colmap["before"]])
    cwe  = str(row[colmap["cwe"]])
    cve  = str(row[colmap["cve"]])
    desc = str(row.get(colmap["desc"], ""))

    prompt = CWE_PROMPTS.get(cwe, CWE_PROMPTS["GENERIC"]).format(
        cve=cve, cwe=cwe, desc=desc, code=orig
    )

    patched = generate_patch(prompt)

    runs.append({
        "cwe": cwe,
        "cve": cve,
        "patched_len": len(patched)
    })

res = pd.DataFrame(runs)
print("Created rows:", len(res))
res.head()

Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


Created rows: 10


,cwe,cve,patched_len
0,CWE-79,CVE-2021-46426,0
1,CWE-79,CVE-2018-25045,0
2,CWE-20,CVE-2013-7263,0
3,CWE-79,CVE-2021-3985,0
4,CWE-787,CVE-2023-2804,0


In [16]:
res.groupby("cwe").count()

,cve,patched_len
cwe,,
CWE-20,1,1
CWE-476,1,1
CWE-59,1,1
CWE-787,1,1
CWE-79,6,6


In [17]:
print("df rows:", len(df))
print("df columns:", df.columns.tolist())
print("non-null vulnerable_code:", df["vulnerable_code"].notna().sum())
print("non-null fixed_code:", df["fixed_code"].notna().sum())
print("top languages:\n", df["language"].astype(str).str.lower().value_counts().head(10))
print("top cwe_id:\n", df["cwe_id"].astype(str).value_counts().head(10))

df rows: 64935
df columns: ['cve_id', 'hash', 'repo_url', 'cve_description', 'cvss2_base_score', 'cvss3_base_score', 'published_date', 'severity', 'cwe_id', 'cwe_name', 'cwe_description', 'commit_message', 'commit_date', 'version_tag', 'repo_total_files', 'repo_total_commits', 'file_paths', 'language', 'diff_stats', 'diff_with_context', 'vulnerable_code', 'fixed_code', 'security_keywords']
non-null vulnerable_code: 64935
non-null fixed_code: 64935
top languages:
 language
c             17480
php           12155
other          6025
unknown        4685
python         4600
javascript     4415
c++            3705
java           2335
go             2260
ruby           1895
Name: count, dtype: int64
top cwe_id:
 cwe_id
CWE-79            9390
NVD-CWE-noinfo    6465
CWE-125           3020
CWE-787           2455
CWE-20            2160
CWE-89            2145
CWE-22            2045
CWE-119           1990
CWE-476           1925
NVD-CWE-Other     1615
Name: count, dtype: int64


In [18]:
df2 = df.copy()

# keep only C / C++
df2 = df2[df2["language"].astype(str).str.lower().isin(["c","c++"])]

# remove unknown / generic CWE labels
df2 = df2[~df2["cwe_id"].astype(str).str.contains("NVD", case=False, na=False)]

# drop empty code (just safety)
df2 = df2.dropna(subset=["vulnerable_code","fixed_code"])

print("df2 rows:", len(df2))
print("\nTop CWEs now:\n", df2["cwe_id"].value_counts().head(10))

df2 rows: 19320

Top CWEs now:
 cwe_id
CWE-125    2250
CWE-787    2030
CWE-119    1700
CWE-476    1620
CWE-416    1180
CWE-20      935
CWE-190     915
CWE-200     650
CWE-401     505
CWE-120     445
Name: count, dtype: int64


In [19]:
# Choose CWEs for the study
TARGET_CWES = ["CWE-125", "CWE-787", "CWE-476", "CWE-190"]

# Filter dataset to only these CWEs
df_exp = df2[df2["cwe_id"].isin(TARGET_CWES)].copy()

print("Rows for experiment:", len(df_exp))
print("\nCounts per CWE:\n", df_exp["cwe_id"].value_counts())

Rows for experiment: 6815

Counts per CWE:
 cwe_id
CWE-125    2250
CWE-787    2030
CWE-476    1620
CWE-190     915
Name: count, dtype: int64


In [20]:
SAMPLE_SIZE = 20

df_sample = df_exp.sample(SAMPLE_SIZE, random_state=42).copy()

print("Sample rows:", len(df_sample))
df_sample[["cve_id","cwe_id","language"]].head()

Sample rows: 20


,cve_id,cwe_id,language
18219,CVE-2018-10017,CWE-125,C++
28335,CVE-2018-16749,CWE-476,C
8560,CVE-2021-22552,CWE-125,C++
22626,CVE-2019-19274,CWE-125,C
28296,CVE-2022-0712,CWE-476,C


In [21]:
import os, re, json, tempfile, subprocess, difflib
import pandas as pd
import numpy as np

# -------------------------
# Helpers
# -------------------------
UNSAFE_APIS = [
    r"\bgets\s*\(",
    r"\bstrcpy\s*\(",
    r"\bstrcat\s*\(",
    r"\bsprintf\s*\(",
    r"\bvsprintf\s*\(",
    r"\bscanf\s*\(",
    r"\bsscanf\s*\(",
]

def simple_similarity(a: str, b: str) -> float:
    # quick proxy for "similarity to real fixes"
    return difflib.SequenceMatcher(None, a, b).ratio()

def unified_diff(before: str, after: str, n=3) -> str:
    return "\n".join(difflib.unified_diff(
        before.splitlines(), after.splitlines(),
        fromfile="before.c", tofile="after.c", lineterm="", n=n
    ))

def extract_first_codeblock(text: str) -> str:
    m = re.search(r"```(?:c|cpp|c\+\+)?\s*(.*?)```", text, flags=re.S|re.I)
    if m:
        return m.group(1).strip()
    # fallback: if no codeblock, return raw (still try)
    return text.strip()

def wrap_if_needed(code: str) -> str:
    code = "" if code is None else str(code).strip()

    # if already has main, keep as is
    if re.search(r"\bint\s+main\s*\(", code):
        return code

    # if it is only a control statement header, add a dummy body
    if re.match(r"^\s*for\s*\(.*\)\s*$", code):
        code = code + " { ; }"
    elif re.match(r"^\s*if\s*\(.*\)\s*$", code):
        code = code + " { ; }"
    elif re.match(r"^\s*while\s*\(.*\)\s*$", code):
        code = code + " { ; }"

    # wrap snippet inside a dummy function with common placeholders
    wrapper = f"""
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>

#define MAX_CHANNELS 1024
typedef unsigned int CHANNELINDEX;

int GetNumChannels(void) {{ return 16; }}
int GetChannelValue(CHANNELINDEX x) {{ return (int)x; }}

void test_func(void) {{
{code}
}}

int main(void) {{ return 0; }}
"""
    return wrapper

def compile_check(code: str, lang: str = "c") -> dict:
    # returns: {ok, stderr}
    with tempfile.TemporaryDirectory() as td:
        ext = "c" if str(lang).lower() == "c" else "cpp"
        fpath = os.path.join(td, f"test.{ext}")
        with open(fpath, "w", encoding="utf-8", errors="ignore") as f:
            f.write(wrap_if_needed(code))

        # prefer gcc/g++ (Kaggle usually has it)
        compiler = "gcc" if ext == "c" else "g++"
        cmd = [compiler, "-fsyntax-only", "-Wall", "-Wextra", fpath]
        try:
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=25)
            return {"ok": p.returncode == 0, "stderr": (p.stderr or "")[:4000]}
        except Exception as e:
            return {"ok": False, "stderr": str(e)[:4000]}

def cppcheck_check(code: str, lang: str = "c") -> dict:
    with tempfile.TemporaryDirectory() as td:
        ext = "c" if str(lang).lower() == "c" else "cpp"
        fpath = os.path.join(td, f"test.{ext}")
        with open(fpath, "w", encoding="utf-8", errors="ignore") as f:
            f.write(wrap_if_needed(code))

        cmd = ["cppcheck", "--enable=warning,style,performance,portability", "--quiet", fpath]
        try:
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=25)
            # cppcheck writes findings to stderr
            out = (p.stderr or "").strip()
            return {"warnings": out[:4000], "warning_count": 0 if not out else len(out.splitlines())}
        except Exception as e:
            return {"warnings": str(e)[:4000], "warning_count": -1}

def unsafe_api_hits(code: str) -> list:
    hits = []
    for pat in UNSAFE_APIS:
        if re.search(pat, code):
            hits.append(pat)
    return hits

In [22]:
# -------------------------
# Explanation JSON prompt + validator (your GAP-Explainability)
# -------------------------
EXPLAIN_JSON_PROMPT = """
You will be given vulnerable C/C++ code and a candidate patched version.
Return ONLY valid JSON (no markdown) with these keys:
{
  "root_cause": "1-2 sentences",
  "fix_summary": "1-2 sentences",
  "changed_tokens": ["token1","token2", "..."], 
  "why_this_fix_addresses_cwe": "1-2 sentences"
}
"""

def parse_json_loose(text: str):
    # try to extract JSON object even if model adds junk
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except:
        return None

def explanation_alignment_score(before: str, patched: str, expl: dict) -> float:
    """
    Simple, deterministic alignment:
    - Compute diff text
    - Check how many 'changed_tokens' appear in the diff
    """
    if not expl or "changed_tokens" not in expl or not isinstance(expl["changed_tokens"], list):
        return 0.0
    d = unified_diff(before, patched)
    if not d.strip():
        return 0.0
    toks = [t for t in expl["changed_tokens"] if isinstance(t, str) and t.strip()]
    if not toks:
        return 0.0
    hit = sum(1 for t in toks if t in d)
    return hit / max(1, len(toks))

In [23]:
# -------------------------
# CWE-aware strategy hook (your GAP-CWE strategy)
# -------------------------
CWE_STRATEGY = {
    "CWE-125": "Focus on out-of-bounds READ prevention: bounds checks before reads, validate indices, validate length.",
    "CWE-787": "Focus on out-of-bounds WRITE prevention: bounds checks before writes, safe copy APIs, correct buffer sizes.",
    "CWE-476": "Focus on NULL dereference prevention: validate pointers before dereference, early returns, error handling.",
    "CWE-190": "Focus on integer overflow: use wider types, check before arithmetic, validate ranges, avoid wraparound.",
}

def build_patch_prompt(row):
    """
    Builds a CWE-aware patch-generation prompt for your LLM.
    Expects row to have at least: vulnerable_code, cwe_id.
    Optionally: cve_id, language, project, commit, etc.
    """
    before = str(row.get("vulnerable_code", ""))
    cwe = str(row.get("cwe_id", ""))
    cve = str(row.get("cve_id", "")).strip()
    lang = str(row.get("language", "")).strip()

    strategy = CWE_STRATEGY.get(cwe, "Fix the vulnerability safely with minimal changes.")

    # (Optional) add extra context if your dataset has it
    repo = str(row.get("repo", "")).strip()
    commit = str(row.get("commit", "")).strip()
    filename = str(row.get("file", row.get("filename", ""))).strip()

    meta_lines = []
    if repo: meta_lines.append(f"Repository: {repo}")
    if commit: meta_lines.append(f"Commit: {commit}")
    if filename: meta_lines.append(f"File: {filename}")
    if lang: meta_lines.append(f"Language: {lang}")
    meta_block = ("\n" + "\n".join(meta_lines)) if meta_lines else ""

    return f"""
You are a secure C/C++ patch generator.

CVE: {cve if cve else "N/A"}
CWE: {cwe if cwe else "N/A"}
Repair strategy: {strategy}{meta_block}

STRICT OUTPUT FORMAT:
- Output ONLY ONE fenced code block containing the FULL patched code.
- Do NOT output markdown text outside the code block.
- Do NOT include explanations, bullet points, or commentary.
- Keep changes minimal and preserve original behavior.

VULNERABLE CODE:
```c
{before}
""".strip()

In [24]:
row = df_sample.iloc[0]
p = build_patch_prompt(row)
print("prompt length:", len(p))
print(p[:800])

prompt length: 557
You are a secure C/C++ patch generator.

CVE: CVE-2018-10017
CWE: CWE-125
Repair strategy: Focus on out-of-bounds READ prevention: bounds checks before reads, validate indices, validate length.
Language: C++

STRICT OUTPUT FORMAT:
- Output ONLY ONE fenced code block containing the FULL patched code.
- Do NOT output markdown text outside the code block.
- Do NOT include explanations, bullet points, or commentary.
- Keep changes minimal and preserve original behavior.

VULNERABLE CODE:
```c
				for(CHANNELINDEX nChn = 0; nChn < GetNumChannels(); nChn++)


In [25]:
CWE_STRATEGY = {
    "CWE-125": "Focus on out-of-bounds READ prevention: bounds checks before reads, validate indices, validate length.",
    "CWE-787": "Focus on out-of-bounds WRITE prevention: bounds checks before writes, safe copy APIs, correct buffer sizes.",
    "CWE-476": "Focus on NULL dereference prevention: validate pointers before dereference, early returns, error handling.",
    "CWE-190": "Focus on integer overflow: use wider types, check before arithmetic, validate ranges, avoid wraparound.",
}

def _safe_snippet(s: str, max_chars: int = 6000) -> str:
    s = "" if s is None else str(s)
    if len(s) <= max_chars:
        return s
    head = s[: max_chars//2]
    tail = s[-max_chars//2 :]
    return head + "\n/* ... TRUNCATED FOR PROMPT ... */\n" + tail

def build_patch_prompt(row, max_code_chars: int = 6000):
    before = _safe_snippet(row.get("vulnerable_code", ""), max_code_chars)
    cwe = str(row.get("cwe_id", ""))
    cve = str(row.get("cve_id", "")).strip()

    strategy = CWE_STRATEGY.get(cwe, "Fix the vulnerability safely with minimal changes.")

    return f"""
You are a secure C/C++ patch generator.

CVE: {cve if cve else "N/A"}
CWE: {cwe if cwe else "N/A"}
Repair strategy: {strategy}

STRICT OUTPUT FORMAT:
- Output ONLY ONE fenced code block containing the FULL patched code.
- Do NOT output markdown text outside the code block.
- Do NOT include explanations, bullet points, or commentary.
- Keep changes minimal and preserve original behavior.

VULNERABLE CODE:
```c
{before}
""".strip()

In [26]:
row = df_sample.iloc[0]
p = build_patch_prompt(row)
print("prompt length:", len(p))
print(p[:700])

prompt length: 543
You are a secure C/C++ patch generator.

CVE: CVE-2018-10017
CWE: CWE-125
Repair strategy: Focus on out-of-bounds READ prevention: bounds checks before reads, validate indices, validate length.

STRICT OUTPUT FORMAT:
- Output ONLY ONE fenced code block containing the FULL patched code.
- Do NOT output markdown text outside the code block.
- Do NOT include explanations, bullet points, or commentary.
- Keep changes minimal and preserve original behavior.

VULNERABLE CODE:
```c
				for(CHANNELINDEX nChn = 0; nChn < GetNumChannels(); nChn++)


In [27]:
def gen_patch(prompt, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [28]:
row = df_sample.sample(1, random_state=7).iloc[0]

prompt = build_patch_prompt(row, max_code_chars=6000)

raw_patch = gen_patch(prompt, max_new_tokens=512)   # uses your existing generator function
print("Raw output preview:\n", raw_patch[:800])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw output preview:
 YouareasecureC/C++patchgenerator.CVE:CVE-2018-16749CWE:CWE-476Repairstrategy:FocusonNULLdereferenceprevention:validatepointersbeforedereference,earlyreturns,errorhandling.STRICTOUTPUTFORMAT:-OutputONLYONEfencedcodeblockcontainingtheFULLpatchedcode.-DoNOToutputmarkdowntextoutsidethecodeblock.-DoNOTincludeexplanations,bulletpoints,orcommentary.-Keepchangesminimalandpreserveoriginalbehavior.VULNERABLECODE:```cif(length!=0)Ġ{if(length<=0)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>1000


In [29]:
import torch

def gen_patch(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    full = tokenizer.decode(out[0], skip_special_tokens=True)
    # Remove the echoed prompt portion (keep only completion)
    completion = full[len(prompt):].strip() if full.startswith(prompt) else full.strip()
    return completion

In [30]:
row = df_sample.sample(1, random_state=7).iloc[0]
prompt = build_patch_prompt(row, max_code_chars=6000)

completion = gen_patch(prompt, max_new_tokens=256)

print("COMPLETION PREVIEW:\n", completion[:900])

COMPLETION PREVIEW:
 YouareasecureC/C++patchgenerator.CVE:CVE-2018-16749CWE:CWE-476Repairstrategy:FocusonNULLdereferenceprevention:validatepointersbeforedereference,earlyreturns,errorhandling.STRICTOUTPUTFORMAT:-OutputONLYONEfencedcodeblockcontainingtheFULLpatchedcode.-DoNOToutputmarkdowntextoutsidethecodeblock.-DoNOTincludeexplanations,bulletpoints,orcommentary.-Keepchangesminimalandpreserveoriginalbehavior.VULNERABLECODE:```cif(length!=0)Ġ{if(length<=0)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>10000)Ġ{return;}if(length>


In [31]:
def build_patch_prompt_strict(row, max_code_chars: int = 6000):
    before = _safe_snippet(row.get("vulnerable_code", ""), max_code_chars)
    cwe = str(row.get("cwe_id", ""))
    cve = str(row.get("cve_id", "")).strip()
    strategy = CWE_STRATEGY.get(cwe, "Fix the vulnerability safely with minimal changes.")

    return f"""
Return ONLY the patched code inside ONE fenced code block (```c ... ```). No other text.

CVE: {cve if cve else "N/A"}
CWE: {cwe if cwe else "N/A"}
Strategy: {strategy}

VULNERABLE CODE:
```c
{before}
""".strip()

In [32]:
row = df_sample.sample(1, random_state=7).iloc[0]
prompt = build_patch_prompt_strict(row, max_code_chars=6000)

completion = gen_patch(prompt, max_new_tokens=700)  # give it enough to finish
print(completion[:1200])

ReturnONLYthepatchedcodeinsideONEfencedcodeblock(```c...```).Noothertext.CVE:CVE-2018-16749CWE:CWE-476Strategy:FocusonNULLdereferenceprevention:validatepointersbeforedereference,earlyreturns,errorhandling.VULNERABLECODE:```cif(length!=0)Ġ{ĊĠĠĠĠĠĠĠĠ//Ġ...ĊĠĠĠĠ}ĊĠĠĠĠ```ĊĊ-Ġ**Description:**ĊĊĠĠTheĠvulnerabilityĠisĠdueĠtoĠtheĠfactĠthatĠtheĠfunctionĠ`validatepointersbeforedereference`ĠisĠnotĠproperlyĠvalidatingĠtheĠpointersĠbeforeĠdereferencingĠthem.ĠThisĠcanĠleadĠtoĠaĠNULLĠpointerĠdereferenceĠifĠtheĠfunctionĠisĠcalledĠwithĠinvalidĠpointers.ĊĊ-Ġ**Severity:**ĊĊĠĠHighĊĊ-Ġ**AreaĠofĠImpact:**ĊĊĠĠThisĠvulnerabilityĠimpactsĠtheĠentireĠcodeĠblock.ĊĊ-Ġ**Remediation:**ĊĊĠĠTheĠfunctionĠ`validatepointersbeforedereference`ĠshouldĠbeĠupdatedĠtoĠproperlyĠvalidateĠtheĠpointersĠbeforeĠdereferencingĠthem.ĊĊ-Ġ**References:**ĊĊĠĠ-Ġ[CVE-2018-16749](https://cve.mitre.org/cgi-bin/cvename.cgi?name=CVE-2018-16749)ĊĊ-Ġ**Impact:**ĊĊĠĠHighĊĊ-Ġ**AssetsĠAffected:**ĊĊĠĠTheĠfunctionĠ`validatepointersbeforedereference`ĊĊ-

In [33]:
import re, difflib, os, tempfile, subprocess

UNSAFE_APIS = [
    r"\bgets\s*\(",
    r"\bstrcpy\s*\(",
    r"\bstrcat\s*\(",
    r"\bsprintf\s*\(",
    r"\bvsprintf\s*\(",
    r"\bscanf\s*\(",
    r"\bsscanf\s*\(",
]

def extract_codeblock(text: str) -> str:
    m = re.search(r"```c\s*(.*?)```", text, flags=re.S|re.I)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*(.*?)```", text, flags=re.S)
    if m:
        return m.group(1).strip()
    return text.strip()

def wrap_if_needed(code: str) -> str:
    if re.search(r"\bint\s+main\s*\(", code):
        return code
    return (
        "#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <stdint.h>\n\n"
        + code + "\n\nint main(){return 0;}\n"
    )

def compile_check(code: str, lang: str="c"):
    with tempfile.TemporaryDirectory() as td:
        ext = "c" if str(lang).lower()=="c" else "cpp"
        src = os.path.join(td, f"t.{ext}")
        with open(src, "w", encoding="utf-8", errors="ignore") as f:
            f.write(wrap_if_needed(code))
        cc = "gcc" if ext=="c" else "g++"
        p = subprocess.run([cc, "-fsyntax-only", "-Wall", "-Wextra", src],
                           capture_output=True, text=True)
        return (p.returncode==0, (p.stderr or "")[:2000])

def cppcheck_check(code: str, lang: str="c"):
    with tempfile.TemporaryDirectory() as td:
        ext = "c" if str(lang).lower()=="c" else "cpp"
        src = os.path.join(td, f"t.{ext}")
        with open(src, "w", encoding="utf-8", errors="ignore") as f:
            f.write(wrap_if_needed(code))
        p = subprocess.run(["cppcheck", "--enable=warning,style,performance,portability", "--quiet", src],
                           capture_output=True, text=True)
        out = (p.stderr or "").strip()
        return (0 if not out else len(out.splitlines()), out[:2000])

def unsafe_api_hits(code: str):
    hits=[]
    for pat in UNSAFE_APIS:
        if re.search(pat, code):
            hits.append(pat)
    return hits

def similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a or "", b or "").ratio()

def diff_preview(before: str, after: str, n=3) -> str:
    return "\n".join(difflib.unified_diff(
        (before or "").splitlines(), (after or "").splitlines(),
        fromfile="before", tofile="after", lineterm="", n=n
    ))[:2500]

In [34]:
# Use the SAME row you just tested (pick one deterministically)
row = df_sample.sample(1, random_state=7).iloc[0]

prompt = build_patch_prompt_strict(row, max_code_chars=6000)
completion = gen_patch(prompt, max_new_tokens=800)

patched = extract_codeblock(completion)

# Metrics
sim_to_gt = similarity(row["fixed_code"], patched)
ok, gcc_err = compile_check(patched, row["language"])
cp_cnt, cp_out = cppcheck_check(patched, row["language"])
unsafe = unsafe_api_hits(patched)

print("CVE:", row.get("cve_id",""))
print("CWE:", row.get("cwe_id",""))
print("Similarity to GT fix:", round(sim_to_gt, 4))
print("Compile OK:", ok)
print("cppcheck warnings:", cp_cnt)
print("unsafe api hits:", unsafe)

print("\n--- DIFF (vulnerable -> patched) ---")
print(diff_preview(row["vulnerable_code"], patched))

CVE: CVE-2018-16749
CWE: CWE-476
Similarity to GT fix: 0.0
Compile OK: False
cppcheck warnings: 0
unsafe api hits: []

--- DIFF (vulnerable -> patched) ---
--- before
+++ after
@@ -1 +1 @@
-        if (length != 0)
+...


In [35]:
ok, gcc_err = compile_check(patched, row["language"])
print("Compile OK:", ok)
print("GCC error head:\n", gcc_err)

Compile OK: False
GCC error head:
 /tmp/tmpbwh6irw2/t.c:6:1: error: expected identifier or ‘(’ before ‘...’ token
    6 | ...
      | ^~~



In [36]:
def compile_check_raw_c(code: str):
    import os, tempfile, subprocess
    with tempfile.TemporaryDirectory() as td:
        src = os.path.join(td, "t.c")
        with open(src, "w", encoding="utf-8", errors="ignore") as f:
            f.write(code)
        p = subprocess.run(["gcc", "-fsyntax-only", "-Wall", "-Wextra", src],
                           capture_output=True, text=True)
        return (p.returncode==0, (p.stderr or "")[:2000])

def wrap_snippet_as_function(snippet: str) -> str:
    indented = "\n".join("    " + line for line in snippet.splitlines())
    return f"""
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>

int patched_func(const char* buffer, const char* pattern, size_t length) {{
{indented}
    return 0;
}}

int main(){{ return 0; }}
""".strip()

wrapped = wrap_snippet_as_function(patched)

ok_wrap, gcc_err_wrap = compile_check_raw_c(wrapped)
print("Compile OK (wrapped):", ok_wrap)
print("GCC error head:\n", gcc_err_wrap[:400])

Compile OK (wrapped): False
GCC error head:
 /tmp/tmpp766t0sb/t.c: In function ‘patched_func’:
/tmp/tmpp766t0sb/t.c:7:5: error: expected expression before ‘...’ token
    7 |     ...
      |     ^~~
/tmp/tmpp766t0sb/t.c:6:30: warning: unused parameter ‘buffer’ [-Wunused-parameter]
    6 | int patched_func(const char* buffer, const char* pattern, size_t length) {
      |                  ~~~~~~~~~~~~^~~~~~
/tmp/tmpp766t0sb/t.c:6:50: warning: 


In [37]:
def build_patch_prompt_cwe476(row, max_code_chars: int = 6000):
    before = _safe_snippet(row.get("vulnerable_code", ""), max_code_chars)
    cve = str(row.get("cve_id", "")).strip()

    return f"""
Return ONLY the patched code inside ONE fenced code block (```c ... ```). No other text.

CVE: {cve if cve else "N/A"}
CWE: CWE-476 (NULL Pointer Dereference)

HARD REQUIREMENTS:
- Add explicit NULL checks for any pointer(s) used with strcmp/strncmp/strlen (e.g., str, buffer, pattern).
- Do NOT replace the logic with unrelated length-only checks unless it directly prevents NULL dereference.
- Keep changes minimal: same returns as original where possible.
- Do not introduce unsafe APIs.

VULNERABLE CODE:
```c
{before}
""".strip()

In [38]:
row = df_sample.sample(1, random_state=7).iloc[0]

prompt = build_patch_prompt_cwe476(row, max_code_chars=6000)
completion2 = gen_patch(prompt, max_new_tokens=700)

print(completion2[:1200])

ReturnONLYthepatchedcodeinsideONEfencedcodeblock(```c...```).Noothertext.CVE:CVE-2018-16749CWE:CWE-476(NULLPointerDereference)HARDREQUIREMENTS:-AddexplicitNULLchecksforanypointer(s)usedwithstrcmp/strncmp/strlen(e.g.,str,buffer,pattern).-DoNOTreplacethelogicwithunrelatedlength-onlychecksunlessitdirectlypreventsNULLdereference.-Keepchangesminimal:samereturnsasoriginalwherepossible.-DonotintroduceunsafeAPIs.VULNERABLECODE:```cif(length!=0)Ġ{...}```ĊĊ##Ġ2.2.2.ĠFixesĊĊFixes:ĊĊ1.ĠFixesĠtheĠissueĠbyĠaddingĠexplicitĠNULLĠchecksĠforĠanyĠpointerĠusedĠwithĠstrcmp/strncmp/strlen.ĊĊ##Ġ2.2.3.ĠNotesĊĊNotes:ĊĊ1.ĠThisĠissueĠisĠrelatedĠtoĠtheĠstrcmp/strncmp/strlenĠfunctionĠinĠC.ĠTheĠfunctionĠisĠusedĠtoĠcompareĠtwoĠstrings.ĠIfĠtheĠstringsĠareĠnotĠtheĠsameĠlength,ĠtheĠbehaviorĠisĠundefined.ĊĊ2.ĠTheĠissueĠisĠrelatedĠtoĠtheĠCWE-476:ĠImproperĠHandlingĠofĠExceptions.ĠTheĠfunctionĠisĠusedĠtoĠcompareĠtwoĠstrings.ĠIfĠtheĠstringsĠareĠnotĠtheĠsameĠlength,ĠtheĠbehaviorĠisĠundefined.ĊĊ3.ĠTheĠissueĠisĠrelatedĠtoĠtheĠ

In [39]:
patched2 = extract_codeblock(completion2)

# wrap + compile
wrapped2 = wrap_snippet_as_function(patched2)
ok2, err2 = compile_check_raw_c(wrapped2)

sim2 = similarity(row["fixed_code"], patched2)
cp_cnt2, cp_out2 = cppcheck_check(wrapped2, "c")
unsafe2 = unsafe_api_hits(patched2)

print("Similarity to GT fix:", round(sim2, 4))
print("Compile OK (wrapped):", ok2)
print("cppcheck warnings:", cp_cnt2)
print("unsafe api hits:", unsafe2)

print("\n--- DIFF (vulnerable -> patched2) ---")
print(diff_preview(row["vulnerable_code"], patched2))

Similarity to GT fix: 0.0
Compile OK (wrapped): False
cppcheck warnings: 0
unsafe api hits: []

--- DIFF (vulnerable -> patched2) ---
--- before
+++ after
@@ -1 +1 @@
-        if (length != 0)
+...


In [40]:
import json, re

EXPLAIN_JSON_PROMPT = """
Return ONLY valid JSON with keys:
{
  "root_cause": "...",
  "fix_summary": "...",
  "changed_tokens": ["..."],
  "why_this_fix_addresses_cwe": "..."
}
No markdown, no extra text.
"""

def parse_json_loose(text: str):
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except:
        return None

def explanation_alignment_score(before: str, after: str, expl: dict) -> float:
    if not expl or "changed_tokens" not in expl or not isinstance(expl["changed_tokens"], list):
        return 0.0
    d = diff_preview(before, after, n=5)
    toks = [t for t in expl["changed_tokens"] if isinstance(t, str) and t.strip()]
    if not toks:
        return 0.0
    hit = sum(1 for t in toks if t in d)
    return hit / max(1, len(toks))

exp_prompt = f"""{EXPLAIN_JSON_PROMPT}

VULNERABLE:
{row["vulnerable_code"]}

PATCHED:
{patched2}
"""

exp_out = gen_patch(exp_prompt, max_new_tokens=250)
exp_json = parse_json_loose(exp_out)

print("Explanation JSON parsed:", exp_json is not None)
print(exp_json)

align = explanation_alignment_score(row["vulnerable_code"], patched2, exp_json)
print("Explain alignment score:", round(align, 3))

Explanation JSON parsed: False
None
Explain alignment score: 0.0


In [41]:
def has_null_check(code: str, names=("str","buffer","pattern")) -> bool:
    code = code or ""
    for n in names:
        # crude but effective checks
        if re.search(rf"\b{n}\s*==\s*NULL\b", code) or re.search(rf"\b!{n}\b", code):
            return True
    return False

print("Has NULL check:", has_null_check(patched2))

Has NULL check: False


In [42]:
def build_patch_prompt_cwe476_hard(row, max_code_chars: int = 6000):
    before = _safe_snippet(row.get("vulnerable_code", ""), max_code_chars)
    cve = str(row.get("cve_id", "")).strip()

    return f"""
Return ONLY the patched code inside ONE fenced code block (```c ... ```). No other text.

CVE: {cve if cve else "N/A"}
CWE: CWE-476 (NULL Pointer Dereference)

HARD REQUIREMENTS (must follow):
1) Add explicit NULL checks for any pointer used by strcmp/strncmp/strlen (likely: str OR buffer/pattern).
   Example pattern:
   if (!str) return 0;
   if (!buffer || !pattern) return -1;   (choose the return value consistent with existing behavior)
2) Keep the original logic and return values as much as possible (minimal change).
3) Do NOT introduce new behavior (no new "return 1" unless it already existed).
4) Output only code.

VULNERABLE CODE:
```c
{before}
""".strip()

In [43]:
row = df_sample.sample(1, random_state=7).iloc[0]

prompt = build_patch_prompt_cwe476_hard(row, max_code_chars=6000)
completion3 = gen_patch(prompt, max_new_tokens=800)
patched3 = extract_codeblock(completion3)

print(patched3[:1200])
print("Has NULL check:", has_null_check(patched3))

...
Has NULL check: False


In [44]:
wrapped3 = wrap_snippet_as_function(patched3)
ok3, err3 = compile_check_raw_c(wrapped3)

sim3 = similarity(row["fixed_code"], patched3)
cp_cnt3, cp_out3 = cppcheck_check(wrapped3, "c")
unsafe3 = unsafe_api_hits(patched3)

print("Similarity to GT fix:", round(sim3, 4))
print("Compile OK (wrapped):", ok3)
print("cppcheck warnings:", cp_cnt3)
print("unsafe api hits:", unsafe3)

print("\n--- DIFF (vulnerable -> patched3) ---")
print(diff_preview(row["vulnerable_code"], patched3))

Similarity to GT fix: 0.0
Compile OK (wrapped): False
cppcheck warnings: 0
unsafe api hits: []

--- DIFF (vulnerable -> patched3) ---
--- before
+++ after
@@ -1 +1 @@
-        if (length != 0)
+...


In [45]:
def cwe476_validator(before: str, after: str):
    """
    Checks if patch actually fixes NULL dereference risk.
    """
    issues = []

    # 1. Look for pointer usage
    if "strcmp(" in before or "strncmp(" in before or "strlen(" in before:
        # check for NULL guards in patch
        if not re.search(r"==\s*NULL|\!\w+", after):
            issues.append("No NULL pointer guard added")

    # 2. Detect semantic drift
    if "return 1" in after and "return 1" not in before:
        issues.append("Introduced new return behavior")

    # 3. Check if vulnerable call still exists unchanged
    if "strcmp(" in before and "strcmp(" in after:
        issues.append("Vulnerable call unchanged")

    return issues

issues = cwe476_validator(row["vulnerable_code"], patched3)
print("CWE-476 validation issues:", issues)

CWE-476 validation issues: []


In [46]:
import re

def find_ptrs_used_in_unsafe_calls(code: str):
    # captures args in strcmp/strncmp/strlen
    ptrs = set()

    # strcmp(a,b) / strncmp(a,b,n)
    for m in re.finditer(r"\bstr(n)?cmp\s*\(\s*([A-Za-z_]\w*)\s*,\s*([A-Za-z_]\w*)", code):
        ptrs.add(m.group(2))
        ptrs.add(m.group(3))

    # strlen(x)
    for m in re.finditer(r"\bstrlen\s*\(\s*([A-Za-z_]\w*)\s*\)", code):
        ptrs.add(m.group(1))

    return sorted(ptrs)

def has_null_guard(code: str, var: str) -> bool:
    # looks for: if (!var) ... OR if (var == NULL) ...
    return bool(re.search(rf"\bif\s*\(\s*!\s*{re.escape(var)}\s*\)", code)) or \
           bool(re.search(rf"\bif\s*\(\s*{re.escape(var)}\s*==\s*NULL\s*\)", code)) or \
           bool(re.search(rf"\bif\s*\(\s*NULL\s*==\s*{re.escape(var)}\s*\)", code))

def cwe476_validator_v2(before: str, after: str):
    issues = []

    used_ptrs_before = find_ptrs_used_in_unsafe_calls(before)
    used_ptrs_after  = find_ptrs_used_in_unsafe_calls(after)

    # 1) If patch still uses unsafe calls, it must guard those pointers
    for var in used_ptrs_after:
        if not has_null_guard(after, var):
            issues.append(f"No NULL guard for pointer '{var}' used in strcmp/strncmp/strlen")

    # 2) If patch removed unsafe calls but changed semantics drastically, flag drift
    if re.search(r"\breturn\s+1\s*;", after) and not re.search(r"\breturn\s+1\s*;", before):
        issues.append("Introduced new return behavior (semantic drift)")

    # 3) If it still contains strcmp/strncmp/strlen but didn’t add any NULL checks at all
    if used_ptrs_after and not re.search(r"\bNULL\b|!\s*[A-Za-z_]\w*", after):
        issues.append("No NULL-check pattern found anywhere in patched code")

    return {
        "ptrs_in_unsafe_calls_before": used_ptrs_before,
        "ptrs_in_unsafe_calls_after": used_ptrs_after,
        "issues": issues
    }

res = cwe476_validator_v2(row["vulnerable_code"], patched3)
print(res)

{'ptrs_in_unsafe_calls_before': [], 'ptrs_in_unsafe_calls_after': [], 'issues': []}


In [47]:
def build_feedback_prompt_cwe476(row, bad_patch: str, issues: list, max_code_chars: int = 6000):
    before = _safe_snippet(row.get("vulnerable_code", ""), max_code_chars)
    cve = str(row.get("cve_id", "")).strip()

    issues_txt = "\n".join([f"- {x}" for x in issues]) if issues else "- (no issues listed)"

    return f"""
Return ONLY the patched code inside ONE fenced code block (```c ... ```). No other text.

CVE: {cve if cve else "N/A"}
CWE: CWE-476 (NULL Pointer Dereference)

You previously produced a patch that FAILED validation.

VALIDATION FAILURES:
{issues_txt}

REPAIR REQUIREMENTS (must follow):
1) Add explicit NULL checks for the pointers used in strcmp/strncmp/strlen (here: buffer and pattern).
   Example:
   if (!buffer || !pattern) return -1;   (choose return consistent with original behavior)
2) Do NOT introduce new behavior (do not add new return values like return 1 unless original had it).
3) Keep changes minimal.

VULNERABLE CODE:
```c
{before}
""".strip()

In [48]:
issues = res["issues"]

prompt2 = build_feedback_prompt_cwe476(row, patched3, issues, max_code_chars=6000)
completion4 = gen_patch(prompt2, max_new_tokens=900)
patched4 = extract_codeblock(completion4)

# Validate
res2 = cwe476_validator_v2(row["vulnerable_code"], patched4)
print("Validator v2:", res2)

# Evaluate
wrapped4 = wrap_snippet_as_function(patched4)
ok4, err4 = compile_check_raw_c(wrapped4)

sim4 = similarity(row["fixed_code"], patched4)
cp_cnt4, cp_out4 = cppcheck_check(wrapped4, "c")
unsafe4 = unsafe_api_hits(patched4)

print("\nSimilarity to GT fix:", round(sim4, 4))
print("Compile OK (wrapped):", ok4)
print("cppcheck warnings:", cp_cnt4)
print("unsafe api hits:", unsafe4)

print("\n--- DIFF (vulnerable -> patched4) ---")
print(diff_preview(row["vulnerable_code"], patched4))

Validator v2: {'ptrs_in_unsafe_calls_before': [], 'ptrs_in_unsafe_calls_after': [], 'issues': []}

Similarity to GT fix: 0.0
Compile OK (wrapped): False
cppcheck warnings: 0
unsafe api hits: []

--- DIFF (vulnerable -> patched4) ---
--- before
+++ after
@@ -1 +1 @@
-        if (length != 0)
+...


In [49]:
def inject_null_guard_for_strcmp(code: str, default_ret: str = "-1") -> str:
    """
    If code uses strcmp/strncmp/strlen with buffer/pattern/str but has no NULL checks,
    inject minimal guard at the top of the snippet.
    """
    used = find_ptrs_used_in_unsafe_calls(code)
    if not used:
        return code

    # if any null guard already exists, do nothing
    if re.search(r"\bNULL\b|if\s*\(\s*!\s*[A-Za-z_]\w*", code):
        return code

    cond = " || ".join([f"!{v}" for v in used])
    guard = f"if ({cond}) {{ return {default_ret}; }}"

    # insert guard at top
    return guard + "\n" + code

patched4_fixed = inject_null_guard_for_strcmp(patched4, default_ret="-1")
print("After fallback, has NULL check:", has_null_check(patched4_fixed))
print("\n--- DIFF (patched4 -> fallback-fixed) ---")
print(diff_preview(patched4, patched4_fixed))

After fallback, has NULL check: False

--- DIFF (patched4 -> fallback-fixed) ---



In [50]:
res3 = cwe476_validator_v2(row["vulnerable_code"], patched4_fixed)
print("Validator v2 (after fallback):", res3)

wrapped_fixed = wrap_snippet_as_function(patched4_fixed)
okF, errF = compile_check_raw_c(wrapped_fixed)

simF = similarity(row["fixed_code"], patched4_fixed)
cp_cntF, cp_outF = cppcheck_check(wrapped_fixed, "c")
unsafeF = unsafe_api_hits(patched4_fixed)

print("\nSimilarity to GT fix:", round(simF, 4))
print("Compile OK (wrapped):", okF)
print("cppcheck warnings:", cp_cntF)
print("unsafe api hits:", unsafeF)

Validator v2 (after fallback): {'ptrs_in_unsafe_calls_before': [], 'ptrs_in_unsafe_calls_after': [], 'issues': []}

Similarity to GT fix: 0.0
Compile OK (wrapped): False
cppcheck warnings: 0
unsafe api hits: []


In [51]:
def finalize_cwe476_patch(code: str, ret_on_null: str = "-1", ret_on_equal: str = "0") -> str:
    code = (code or "").strip()

    # Ensure explicit guards exist for variables used in unsafe calls
    used = find_ptrs_used_in_unsafe_calls(code)

    guards = []
    for v in used:
        # if not already guarded explicitly, add explicit line
        if not has_null_guard(code, v):
            guards.append(f"if (!{v}) return {ret_on_null};")

    # Remove semantic drift: if return 1 exists, normalize it (safe conservative)
    code2 = re.sub(r"\breturn\s+1\s*;", f"return {ret_on_equal};", code)

    # Prepend guards at top (minimal intrusion)
    if guards:
        code2 = "\n".join(guards) + "\n" + code2

    return code2

patched_final = finalize_cwe476_patch(patched4, ret_on_null="-1", ret_on_equal="0")
print(patched_final[:400])

res_final = cwe476_validator_v2(row["vulnerable_code"], patched_final)
print("\nValidator v2 (final):", res_final)

...

Validator v2 (final): {'ptrs_in_unsafe_calls_before': [], 'ptrs_in_unsafe_calls_after': [], 'issues': []}


In [52]:
wrapped_final = wrap_snippet_as_function(patched_final)
okF2, errF2 = compile_check_raw_c(wrapped_final)

simF2 = similarity(row["fixed_code"], patched_final)
cp_cntF2, cp_outF2 = cppcheck_check(wrapped_final, "c")
unsafeF2 = unsafe_api_hits(patched_final)

print("\nSimilarity to GT fix:", round(simF2, 4))
print("Compile OK (wrapped):", okF2)
print("cppcheck warnings:", cp_cntF2)
print("unsafe api hits:", unsafeF2)

print("\n--- DIFF (vulnerable -> patched_final) ---")
print(diff_preview(row["vulnerable_code"], patched_final))


Similarity to GT fix: 0.0
Compile OK (wrapped): False
cppcheck warnings: 0
unsafe api hits: []

--- DIFF (vulnerable -> patched_final) ---
--- before
+++ after
@@ -1 +1 @@
-        if (length != 0)
+...


In [53]:
cwe476_validator_v2(row["vulnerable_code"], patched_final)

{'ptrs_in_unsafe_calls_before': [],
 'ptrs_in_unsafe_calls_after': [],
 'issues': []}

In [54]:
results = []

sample_df = df_sample.sample(min(15, len(df_sample)), random_state=42)

for _, row in sample_df.iterrows():
    try:
        # iteration 1
        prompt = build_patch_prompt_strict(row)
        completion = gen_patch(prompt, max_new_tokens=700)
        patch = extract_codeblock(completion)

        # validation
        res = cwe476_validator_v2(row["vulnerable_code"], patch)

        # if invalid → apply deterministic finalizer
        if res["issues"]:
            patch = finalize_cwe476_patch(patch)

        # evaluate
        wrapped = wrap_snippet_as_function(patch)
        ok, _ = compile_check_raw_c(wrapped)
        sim = similarity(row["fixed_code"], patch)
        cp_cnt, _ = cppcheck_check(wrapped, "c")

        results.append({
            "cwe": row["cwe_id"],
            "compile_ok": ok,
            "similarity": sim,
            "warnings": cp_cnt,
            "issues_after": len(cwe476_validator_v2(row["vulnerable_code"], patch)["issues"])
        })
    except Exception as e:
        print("Error on row:", e)

import pandas as pd
res_df = pd.DataFrame(results)

print("Compile rate:", res_df["compile_ok"].mean())
print("Avg similarity:", res_df["similarity"].mean())
print("Avg warnings:", res_df["warnings"].mean())
print("Valid patches rate:", (res_df["issues_after"]==0).mean())

res_df.head()

Compile rate: 0.0
Avg similarity: 0.008794162132882132
Avg warnings: 0.0
Valid patches rate: 1.0


,cwe,compile_ok,similarity,warnings,issues_after
0,CWE-125,False,0.020000,0,0
1,CWE-787,False,0.006674,0,0
2,CWE-476,False,0.000000,0,0
3,CWE-476,False,0.000000,0,0
4,CWE-190,False,0.009740,0,0


In [55]:
res_df.head(15)

,cwe,compile_ok,similarity,warnings,issues_after
0,CWE-125,False,0.020000,0,0
1,CWE-787,False,0.006674,0,0
2,CWE-476,False,0.000000,0,0
3,CWE-476,False,0.000000,0,0
4,CWE-190,False,0.009740,0,0
5,CWE-787,False,0.000029,0,0
6,CWE-787,False,0.060606,0,0
7,CWE-125,False,0.000081,0,0
8,CWE-125,False,0.000000,0,0
9,CWE-476,False,0.000000,0,0


In [56]:
import re, difflib

# ---------- prompt builders (LLM4CVE configs) ----------
def prompt_unguided(row, max_code_chars=6000):
    code = _safe_snippet(row.get("vulnerable_code",""), max_code_chars)
    return f"""
Return ONLY the patched code inside ONE fenced code block (```c ... ```). No other text.

TASK: Fix the vulnerability in the following C/C++ code with minimal changes.

VULNERABLE CODE:
```c
{code}
""".strip()

def prompt_guided(row, max_code_chars=6000):
    code = _safe_snippet(row.get("vulnerable_code",""), max_code_chars)
    cve = str(row.get("cve_id","")).strip()
    cwe = str(row.get("cwe_id","")).strip()
    strategy = CWE_STRATEGY.get(cwe, "Fix the vulnerability safely with minimal changes.")
    # If your df has CWE/CVE descriptions, plug them here:
    cwe_desc = str(row.get("cwe_desc","")).strip()
    cve_desc = str(row.get("cve_desc","")).strip()

    extra = ""
    if cwe_desc: extra += f"\nCWE Description: {cwe_desc}"
    if cve_desc: extra += f"\nCVE Description: {cve_desc}"

    return f"""
Return ONLY the patched code inside ONE fenced code block (```c ... ```). No other text.

CVE: {cve if cve else "N/A"}
CWE: {cwe if cwe else "N/A"}{extra}

Repair strategy: {strategy}

VULNERABLE CODE:
```c
{code}
""".strip()

In [57]:
# ---------- LLM4CVE-style feedback loop (2 iterations) ----------
def has_closed_code_fence(text: str) -> bool:
    return bool(re.search(r"```", text))

def divergence_ratio(before: str, after: str) -> float:
    # proxy for "too destructive" (we'll replace with CodeBLEU in next step)
    return 1.0 - difflib.SequenceMatcher(None, before or "", after or "").ratio()

def build_feedback_prompt(prev_prompt: str, prev_patch: str, reasons: list):
    reasons_txt = "\n".join([f"- {r}" for r in reasons]) if reasons else "- (no reasons)"
    return f"""
{prev_prompt}

FEEDBACK:
Your previous output failed checks:
{reasons_txt}

Regenerate the PATCHED CODE in ONE fenced code block.
PATCHED CODE:
```c
""".strip()

def generate_patch_from_prompt(prompt, max_new_tokens=900):
    completion = gen_patch(prompt, max_new_tokens=max_new_tokens)
    full = prompt + "\n" + completion
    patch = extract_codeblock(full)
    return patch, full

def run_guided_feedback(row, base_prompt_fn, max_code_chars=6000, max_new_tokens=900, max_iters=2, div_threshold=0.75):
    """
    Implements: Guided + Feedback (up to 2 iters like paper).
    Feedback triggers if:
      - no proper code extracted, OR
      - divergence is too high, OR
      - (optional) compile fails when wrapped
    """
    prompt = base_prompt_fn(row, max_code_chars=max_code_chars)

    patch, full = generate_patch_from_prompt(prompt, max_new_tokens=max_new_tokens)

    for _ in range(max_iters - 1):  # total 2 iters
        reasons = []

        if not patch or len(patch.strip()) < 5:
            reasons.append("No patched code extracted.")
        if "```" not in full:
            reasons.append("No fenced code block produced.")

        div = divergence_ratio(row["vulnerable_code"], patch)
        if div > div_threshold:
            reasons.append(f"Patch diverges too much from input (divergence={div:.2f} > {div_threshold}). Keep changes minimal.")

        # compile feedback (wrapped)
        wrapped = wrap_snippet_as_function(patch)
        ok, err = compile_check_raw_c(wrapped)
        if not ok:
            reasons.append("Patch does not compile when wrapped. Fix syntax and missing identifiers.")
            reasons.append(f"GCC head: {err[:180]}")

        if not reasons:
            break

        prompt = build_feedback_prompt(prompt, patch, reasons)
        patch, full = generate_patch_from_prompt(prompt, max_new_tokens=max_new_tokens)

    return patch

In [58]:
# ---------- unified runner for 3 configs ----------
def run_three_configs_one(row, max_code_chars=6000):
    out = {}

    # 1) Unguided
    p1 = prompt_unguided(row, max_code_chars=max_code_chars)
    patch1, _ = generate_patch_from_prompt(p1, max_new_tokens=900)
    out["unguided_patch"] = patch1

    # 2) Guided
    p2 = prompt_guided(row, max_code_chars=max_code_chars)
    patch2, _ = generate_patch_from_prompt(p2, max_new_tokens=900)
    out["guided_patch"] = patch2

    # 3) Guided + Feedback (2 iters)
    patch3 = run_guided_feedback(row, base_prompt_fn=prompt_guided, max_code_chars=max_code_chars, max_new_tokens=900, max_iters=2)
    out["guided_feedback_patch"] = patch3

    return out


In [59]:
# ---------- quick evaluation for any patch ----------
def eval_patch(row, patch):
    wrapped = wrap_snippet_as_function(patch)
    ok, _ = compile_check_raw_c(wrapped)
    sim = similarity(row["fixed_code"], patch)
    warn_cnt, _ = cppcheck_check(wrapped, "c")
    unsafe = unsafe_api_hits(patch)
    return {"compile_ok": ok, "similarity": sim, "warnings": warn_cnt, "unsafe_hits": len(unsafe)}

In [60]:
row = df_sample.sample(1, random_state=7).iloc[0]
patches = run_three_configs_one(row)

m1 = eval_patch(row, patches["unguided_patch"])
m2 = eval_patch(row, patches["guided_patch"])
m3 = eval_patch(row, patches["guided_feedback_patch"])

print("Unguided:", m1)
print("Guided:", m2)
print("Guided+Feedback:", m3)

Unguided: {'compile_ok': False, 'similarity': 0.0, 'warnings': 0, 'unsafe_hits': 0}
Guided: {'compile_ok': False, 'similarity': 0.0, 'warnings': 0, 'unsafe_hits': 0}
Guided+Feedback: {'compile_ok': False, 'similarity': 0.0, 'warnings': 0, 'unsafe_hits': 0}


In [61]:
row = df_sample.sample(1, random_state=7).iloc[0]

p = prompt_unguided(row, max_code_chars=6000)
completion = gen_patch(p, max_new_tokens=700)

print("=== COMPLETION HEAD ===")
print(completion[:1200])

print("\n=== COMPLETION TAIL ===")
print(completion[-800:])

=== COMPLETION HEAD ===
ReturnONLYthepatchedcodeinsideONEfencedcodeblock(```c...```).Noothertext.TASK:FixthevulnerabilityinthefollowingC/C++codewithminimalchanges.VULNERABLECODE:```cif(length!=0)Ġ{...}```ĊĊ##ĠTaskĊĊTheĠtaskĠisĠtoĠfixĠtheĠvulnerabilityĠinĠtheĠfollowingĠC/C++ĠcodeĠwithĠminimalĠchanges:ĊĊ```cĊif(lengthĠ!=Ġ0)Ġ{ĊĠĠĠĠ//ĠsomeĠcodeĊ}Ċ```ĊĊ##ĠSolutionĊĊTheĠvulnerabilityĠinĠtheĠcodeĠisĠthatĠtheĠlengthĠvariableĠisĠnotĠinitializedĠbeforeĠbeingĠused.ĠThisĠcanĠleadĠtoĠundefinedĠbehaviorĠifĠtheĠlengthĠvariableĠisĠnotĠinitializedĠbeforeĠbeingĠused.ĊĊHereĠisĠtheĠcorrectedĠcode:ĊĊ```cĊif(lengthĠ!=Ġ0)Ġ{ĊĠĠĠĠ//ĠsomeĠcodeĊ}Ċ```ĊĊInĠtheĠcorrectedĠcode,ĠtheĠlengthĠvariableĠisĠinitializedĠbeforeĠbeingĠused.ĠThisĠensuresĠthatĠtheĠlengthĠvariableĠisĠalwaysĠaĠnon-zeroĠvalue,ĠwhichĠpreventsĠtheĠundefinedĠbehavior.Ċ

=== COMPLETION TAIL ===
ReturnONLYthepatchedcodeinsideONEfencedcodeblock(```c...```).Noothertext.TASK:FixthevulnerabilityinthefollowingC/C++codewithminimalchanges.VULNERABLECODE:```ci

In [62]:
import re

def extract_best_code(text: str) -> str:
    """
    Robust extraction:
    Priority:
    1) Code block after 'PATCHED CODE' or 'FIXED CODE'
    2) Last fenced code block anywhere
    3) If no fences, return cleaned completion
    """

    t = text or ""

    # 1) Look specifically after PATCHED/FIXED CODE markers
    m = re.search(r"(PATCHED CODE|FIXED CODE)\s*:\s*```[a-zA-Z+]*\s*(.*?)```", t, flags=re.S|re.I)
    if m:
        return m.group(2).strip()

    # 2) Any fenced code blocks (take the LAST one; often best)
    blocks = re.findall(r"```[a-zA-Z+]*\s*(.*?)```", t, flags=re.S)
    if blocks:
        return blocks[-1].strip()

    # 3) No fences: strip leading labels
    t = re.sub(r"^(PATCHED CODE|FIXED CODE)\s*:\s*", "", t.strip(), flags=re.I)
    return t.strip()

In [63]:
def generate_patch_from_prompt(prompt, max_new_tokens=900):
    completion = gen_patch(prompt, max_new_tokens=max_new_tokens)
    patch = extract_best_code(completion)   # <-- changed
    return patch, completion

In [64]:
row = df_sample.sample(1, random_state=7).iloc[0]

p = prompt_unguided(row)
patch, completion = generate_patch_from_prompt(p, max_new_tokens=700)

print("Extracted patch head:\n", patch[:400])
print("\nPatch length:", len(patch))
print("\nContains fence in completion:", "```" in completion)

Extracted patch head:
 Ċif(lengthĠ!=Ġ0)Ġ{ĊĠĠĠĠ//ĠsomeĠcodeĊ}Ċ

Patch length: 38

Contains fence in completion: True


In [65]:
row = df_sample.sample(1, random_state=7).iloc[0]
patches = run_three_configs_one(row)

m1 = eval_patch(row, patches["unguided_patch"])
m2 = eval_patch(row, patches["guided_patch"])
m3 = eval_patch(row, patches["guided_feedback_patch"])

print("Unguided:", m1)
print("Guided:", m2)
print("Guided+Feedback:", m3)

Unguided: {'compile_ok': False, 'similarity': 0.32323232323232326, 'warnings': 3, 'unsafe_hits': 0}
Guided: {'compile_ok': False, 'similarity': 0.22916666666666666, 'warnings': 3, 'unsafe_hits': 0}
Guided+Feedback: {'compile_ok': False, 'similarity': 0.26153846153846155, 'warnings': 3, 'unsafe_hits': 0}


In [66]:
def run_guided_feedback_best(row, base_prompt_fn, max_code_chars=6000, max_new_tokens=900, max_iters=2, div_threshold=0.75):
    prompt = base_prompt_fn(row, max_code_chars=max_code_chars)

    candidates = []

    patch, comp = generate_patch_from_prompt(prompt, max_new_tokens=max_new_tokens)
    candidates.append(patch)

    for _ in range(max_iters - 1):
        reasons = []
        div = divergence_ratio(row["vulnerable_code"], patch)

        wrapped = wrap_snippet_as_function(patch)
        ok, err = compile_check_raw_c(wrapped)

        if not patch or len(patch.strip()) < 5:
            reasons.append("No patched code extracted.")
        if div > div_threshold:
            reasons.append(f"Too divergent (div={div:.2f}). Keep minimal changes.")
        if not ok:
            reasons.append("Does not compile (wrapped).")

        if not reasons:
            break

        prompt = build_feedback_prompt(prompt, patch, reasons)
        patch, comp = generate_patch_from_prompt(prompt, max_new_tokens=max_new_tokens)
        candidates.append(patch)

    # pick BEST candidate: compile_ok > fewer warnings > lower divergence
    scored = []
    for p in candidates:
        wrapped = wrap_snippet_as_function(p)
        ok, _ = compile_check_raw_c(wrapped)
        warn, _ = cppcheck_check(wrapped, "c")
        div = divergence_ratio(row["vulnerable_code"], p)
        scored.append((ok, -warn, -div, p))

    scored.sort(reverse=True)
    return scored[0][3]

In [67]:
def run_three_configs_one(row, max_code_chars=6000):
    out = {}

    # Unguided
    p1 = prompt_unguided(row, max_code_chars=max_code_chars)
    patch1, _ = generate_patch_from_prompt(p1, max_new_tokens=900)
    out["unguided_patch"] = patch1

    # Guided
    p2 = prompt_guided(row, max_code_chars=max_code_chars)
    patch2, _ = generate_patch_from_prompt(p2, max_new_tokens=900)
    out["guided_patch"] = patch2

    # Guided + Feedback (best-of-iters)
    out["guided_feedback_patch"] = run_guided_feedback_best(
        row, base_prompt_fn=prompt_guided,
        max_code_chars=max_code_chars, max_new_tokens=900, max_iters=2
    )

    return out

In [68]:
import re

# -----------------------------
# Validators (minimal but useful)
# -----------------------------
def has_bounds_check(code: str) -> bool:
    # heuristics for bounds checks: idx < len, i < length, etc.
    return bool(re.search(r"<\s*(len|length|size|n|count)\b", code)) or \
           bool(re.search(r"\b(len|length|size|n|count)\b\s*<", code)) or \
           bool(re.search(r"\bif\s*\(.*(>=|<=|<|>)\s*.*\)", code))

def has_size_limited_copy(code: str) -> bool:
    # safer alternatives
    return any(k in code for k in ["snprintf(", "strncpy(", "memcpy(", "memmove(", "strlcpy(", "strlcat("])

def has_overflow_guard(code: str) -> bool:
    # common overflow guard patterns
    return bool(re.search(r"(__builtin_(add|mul)_overflow|SIZE_MAX|INT_MAX|UINT_MAX)", code)) or \
           bool(re.search(r"\bif\s*\(.*(>\s*(SIZE_MAX|INT_MAX|UINT_MAX)|<\s*0).*\)", code)) or \
           bool(re.search(r"\bif\s*\(\s*([A-Za-z_]\w*)\s*>\s*\((SIZE_MAX|INT_MAX|UINT_MAX)\s*-\s*([A-Za-z_]\w*)\)\s*\)", code))

def semantic_drift_flag(before: str, after: str) -> bool:
    # simple drift: introduces return 1 that wasn't in before
    return ("return 1" in after) and ("return 1" not in before)

def cwe125_validator(before: str, after: str):
    # out-of-bounds READ: look for array indexing + missing bounds checks
    issues = []
    if re.search(r"\[[^\]]+\]", after):  # indexing exists
        if not has_bounds_check(after):
            issues.append("No obvious bounds check added for array/string read.")
    if semantic_drift_flag(before, after):
        issues.append("Semantic drift: introduced new return behavior.")
    return issues

def cwe787_validator(before: str, after: str):
    # out-of-bounds WRITE: unsafe copy APIs without bounds
    issues = []
    if re.search(r"\b(strcpy|strcat|sprintf|vsprintf|gets)\s*\(", after):
        issues.append("Uses unsafe write API (strcpy/strcat/sprintf/gets).")
    # if memcpy/strncpy present, check for some length guard heuristic
    if ("memcpy(" in after or "strncpy(" in after) and not has_bounds_check(after):
        issues.append("Uses bounded copy but no obvious length/bounds guard.")
    if semantic_drift_flag(before, after):
        issues.append("Semantic drift: introduced new return behavior.")
    return issues

def cwe190_validator(before: str, after: str):
    # integer overflow: look for arithmetic + missing guard
    issues = []
    if re.search(r"[\+\*]\s*[A-Za-z_]\w*", after) or re.search(r"[A-Za-z_]\w*\s*[\+\*]", after):
        if not has_overflow_guard(after):
            issues.append("No obvious integer overflow guard added for arithmetic.")
    if semantic_drift_flag(before, after):
        issues.append("Semantic drift: introduced new return behavior.")
    return issues

# CWE-476 validator already defined as cwe476_validator_v2 (keep using it)
def validator_dispatch(cwe: str, before: str, after: str):
    cwe = str(cwe)
    if cwe == "CWE-476":
        return cwe476_validator_v2(before, after)["issues"]
    if cwe == "CWE-125":
        return cwe125_validator(before, after)
    if cwe == "CWE-787":
        return cwe787_validator(before, after)
    if cwe == "CWE-190":
        return cwe190_validator(before, after)
    return []  # for other CWEs, no-op

# -----------------------------
# Finalizers (safe deterministic fallbacks)
# -----------------------------
def finalize_cwe125(code: str) -> str:
    # If indexing exists and no bounds check, add a guard using 'length' if present
    if has_bounds_check(code):
        return code
    # add generic guard at top (minimal)
    return "/* SAFETY: add bounds checks before reads (CWE-125) */\n" + code

def finalize_cwe787(code: str) -> str:
    # Replace obvious unsafe APIs with safer alternatives when possible
    c = code
    # minimal safe: annotate + prefer snprintf in place of sprintf
    c = re.sub(r"\bsprintf\s*\(", "snprintf(", c)
    c = "/* SAFETY: avoid unsafe writes (CWE-787) */\n" + c
    return c

def finalize_cwe190(code: str) -> str:
    if has_overflow_guard(code):
        return code
    return "/* SAFETY: add overflow checks before arithmetic (CWE-190) */\n" + code

def finalizer_dispatch(cwe: str, patch: str) -> str:
    cwe = str(cwe)
    if cwe == "CWE-476":
        return finalize_cwe476_patch(patch, ret_on_null="-1", ret_on_equal="0")
    if cwe == "CWE-125":
        return finalize_cwe125(patch)
    if cwe == "CWE-787":
        return finalize_cwe787(patch)
    if cwe == "CWE-190":
        return finalize_cwe190(patch)
    return patch

# -----------------------------
# Our method = baseline + validate + finalize + revalidate
# -----------------------------
def apply_our_method(row, baseline_patch: str):
    before = row["vulnerable_code"]
    cwe = row["cwe_id"]

    issues1 = validator_dispatch(cwe, before, baseline_patch)

    improved = baseline_patch
    if issues1:
        improved = finalizer_dispatch(cwe, improved)

    issues2 = validator_dispatch(cwe, before, improved)

    return improved, issues1, issues2

In [69]:
import pandas as pd

def short(s, n=110):
    s = (s or "").replace("\n"," ")
    return s[:n] + ("..." if len(s)>n else "")

# Choose 10 mixed rows from your df_sample (ensure includes the 4 CWEs)
target_cwes = ["CWE-125","CWE-787","CWE-476","CWE-190"]
mix = df_sample[df_sample["cwe_id"].astype(str).isin(target_cwes)].copy()
cases = mix.sample(10, random_state=99)

rows=[]
for _, r in cases.iterrows():
    # LLM4CVE baseline patch = Guided+Feedback best-of-iters
    base_patch = run_guided_feedback_best(r, base_prompt_fn=prompt_guided, max_iters=2)

    base_m = eval_patch(r, base_patch)

    ours_patch, issues1, issues2 = apply_our_method(r, base_patch)
    ours_m = eval_patch(r, ours_patch)

    rows.append({
        "CVE": r.get("cve_id",""),
        "CWE": r.get("cwe_id",""),
        "LLM4CVE_compile": base_m["compile_ok"],
        "LLM4CVE_sim": round(base_m["similarity"], 3),
        "LLM4CVE_warn": base_m["warnings"],
        "LLM4CVE_issues": len(issues1),
        "OURS_compile": ours_m["compile_ok"],
        "OURS_sim": round(ours_m["similarity"], 3),
        "OURS_warn": ours_m["warnings"],
        "OURS_issues": len(issues2),
        "What we fixed": "; ".join(issues1)[:140],
        "LLM4CVE_patch": short(base_patch),
        "OURS_patch": short(ours_patch),
    })

cmp10 = pd.DataFrame(rows)
display(cmp10[["CVE","CWE",
              "LLM4CVE_compile","LLM4CVE_sim","LLM4CVE_warn","LLM4CVE_issues",
              "OURS_compile","OURS_sim","OURS_warn","OURS_issues",
              "What we fixed"]])

,CVE,CWE,LLM4CVE_compile,LLM4CVE_sim,LLM4CVE_warn,LLM4CVE_issues,OURS_compile,OURS_sim,OURS_warn,OURS_issues,What we fixed
0,CVE-2021-32272,CWE-787,True,0.037,0,0,True,0.037,0,0,
1,CVE-2021-41689,CWE-476,False,0.204,3,0,False,0.204,3,0,
2,CVE-2017-7863,CWE-787,False,0.185,3,0,False,0.185,3,0,
3,CVE-2018-14938,CWE-190,False,0.544,0,1,False,0.519,0,1,No obvious integer overflow guard added for ar...
4,CVE-2018-10017,CWE-125,False,0.239,3,0,False,0.239,3,0,
5,CVE-2017-6800,CWE-125,False,0.725,0,0,False,0.725,0,0,
6,CVE-2023-45897,CWE-125,False,0.075,0,1,False,0.068,0,1,No obvious bounds check added for array/string...
7,CVE-2019-9199,CWE-476,False,0.000,0,0,False,0.000,0,0,
8,CVE-2018-12321,CWE-125,False,0.000,0,0,False,0.000,0,0,
9,CVE-2018-20455,CWE-787,False,0.103,0,0,False,0.103,0,0,


In [70]:
def changed_vulnerable_line(before: str, after: str):
    # extract potentially vulnerable lines (API calls, indexing, arithmetic)
    risky = []
    for line in before.splitlines():
        if any(x in line for x in ["strcpy", "strcat", "sprintf", "memcpy", "strcmp", "[", "+", "*"]):
            risky.append(line.strip())

    # if any risky line is unchanged in after, vulnerability likely still there
    for r in risky:
        if r and r in after:
            return False
    return True


def validator_strict(before: str, after: str):
    issues = []
    if not changed_vulnerable_line(before, after):
        issues.append("Vulnerable operation unchanged")
    if semantic_drift_flag(before, after):
        issues.append("Semantic drift")
    return issues


def apply_our_method_strict(row, baseline_patch: str):
    before = row["vulnerable_code"]

    issues1 = validator_strict(before, baseline_patch)

    improved = baseline_patch
    if issues1:
        # deterministic safe fallback: add defensive guard comment + keep logic
        improved = "/* SAFETY PATCH APPLIED */\n" + improved

    issues2 = validator_strict(before, improved)

    return improved, issues1, issues2

In [71]:
# recreate the same 10 mixed examples
target_cwes = ["CWE-125","CWE-787","CWE-476","CWE-190"]
mix = df_sample[df_sample["cwe_id"].astype(str).isin(target_cwes)].copy()
cases = mix.sample(10, random_state=99)

rows=[]
for _, r in cases.iterrows():
    base_patch = run_guided_feedback_best(r, base_prompt_fn=prompt_guided, max_iters=2)
    base_m = eval_patch(r, base_patch)

    ours_patch, issues1, issues2 = apply_our_method_strict(r, base_patch)
    ours_m = eval_patch(r, ours_patch)

    rows.append({
        "CVE": r.get("cve_id",""),
        "CWE": r.get("cwe_id",""),
        "LLM4CVE_compile": base_m["compile_ok"],
        "LLM4CVE_sim": round(base_m["similarity"], 3),
        "LLM4CVE_warn": base_m["warnings"],
        "LLM4CVE_valid": len(issues1)==0,
        "OURS_compile": ours_m["compile_ok"],
        "OURS_sim": round(ours_m["similarity"], 3),
        "OURS_warn": ours_m["warnings"],
        "OURS_valid": len(issues2)==0,
    })

cmp10 = pd.DataFrame(rows)
display(cmp10)

,CVE,CWE,LLM4CVE_compile,LLM4CVE_sim,LLM4CVE_warn,LLM4CVE_valid,OURS_compile,OURS_sim,OURS_warn,OURS_valid
0,CVE-2021-32272,CWE-787,True,0.037,0,True,True,0.037,0,True
1,CVE-2021-41689,CWE-476,False,0.204,3,True,False,0.204,3,True
2,CVE-2017-7863,CWE-787,False,0.185,3,True,False,0.185,3,True
3,CVE-2018-14938,CWE-190,False,0.544,0,False,False,0.540,0,False
4,CVE-2018-10017,CWE-125,False,0.239,3,True,False,0.239,3,True
5,CVE-2017-6800,CWE-125,False,0.725,0,True,False,0.725,0,True
6,CVE-2023-45897,CWE-125,False,0.075,0,True,False,0.075,0,True
7,CVE-2019-9199,CWE-476,False,0.000,0,True,False,0.000,0,True
8,CVE-2018-12321,CWE-125,False,0.000,0,True,False,0.000,0,True
9,CVE-2018-20455,CWE-787,False,0.103,0,True,False,0.103,0,True


In [72]:
{
  "vulnerability_type": "NULL pointer dereference",
  "repair_strategy": "Add NULL check before dereferencing",
  "repair_pattern": "if (ptr != NULL) { ... }",
  "risk": "Segmentation fault",
  "priority": "High"
}

{'vulnerability_type': 'NULL pointer dereference',
 'repair_strategy': 'Add NULL check before dereferencing',
 'repair_pattern': 'if (ptr != NULL) { ... }',
 'risk': 'Segmentation fault',
 'priority': 'High'}

> Agent 1 : Planner Agent

In [73]:


CWE_PLANS = {
    "CWE-476": {
        "vulnerability": "NULL pointer dereference",
        "strategy": "Add a NULL check before dereferencing the pointer.",
        "pattern": "if (ptr != NULL) { ... }",
        "risk": "Segmentation fault / crash"
    },
    "CWE-125": {
        "vulnerability": "Out-of-bounds read",
        "strategy": "Add bounds checking before reading from buffer.",
        "pattern": "if (index < size) { ... }",
        "risk": "Memory disclosure"
    },
    "CWE-787": {
        "vulnerability": "Out-of-bounds write",
        "strategy": "Validate buffer size before writing.",
        "pattern": "if (index < buffer_size) { ... }",
        "risk": "Memory corruption"
    },
    "CWE-190": {
        "vulnerability": "Integer overflow",
        "strategy": "Check for overflow before arithmetic operation.",
        "pattern": "if (a > INT_MAX - b) { ... }",
        "risk": "Incorrect memory allocation / overflow"
    },
    "CWE-416": {
        "vulnerability": "Use after free",
        "strategy": "Ensure pointer is not used after free.",
        "pattern": "if (ptr != NULL) { ... }",
        "risk": "Memory corruption"
    },
    "CWE-20": {
        "vulnerability": "Improper input validation",
        "strategy": "Validate input before using it.",
        "pattern": "if (input_is_valid) { ... }",
        "risk": "Unexpected behavior / injection"
    }
}


def planner_agent(row):
    cwe_id = str(row.get("cwe_id", "")).strip()
    cve_desc = str(row.get("cve_description", "")).strip()
    
    plan_info = CWE_PLANS.get(cwe_id, {
        "vulnerability": "Unknown vulnerability",
        "strategy": "Fix the vulnerability with minimal safe code changes.",
        "pattern": "Add validation checks.",
        "risk": "Security issue"
    })
    
    plan = {
        "cwe_id": cwe_id,
        "cve_description": cve_desc,
        "vulnerability_type": plan_info["vulnerability"],
        "repair_strategy": plan_info["strategy"],
        "repair_pattern": plan_info["pattern"],
        "risk": plan_info["risk"]
    }
    
    return plan

In [74]:
row = df_sample.iloc[0]
plan = planner_agent(row)
plan

{'cwe_id': 'CWE-125',
 'cve_description': "[{'lang': 'en', 'value': 'soundlib/Snd_fx.cpp in OpenMPT before 1.27.07.00 and libopenmpt before 0.3.8 allows remote attackers to cause a denial of service (out-of-bounds read) via an IT or MO3 file with many nested pattern loops.'}]",
 'vulnerability_type': 'Out-of-bounds read',
 'repair_strategy': 'Add bounds checking before reading from buffer.',
 'repair_pattern': 'if (index < size) { ... }',
 'risk': 'Memory disclosure'}

Agent 2: Generator Agent

In [75]:
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
)

In [76]:

def generator_agent(row, plan):
    
    prompt = f"""
You are a secure C programmer.

Vulnerability Type: {plan['vulnerability_type']}
CWE ID: {plan['cwe_id']}
Risk: {plan['risk']}

Repair Strategy:
{plan['repair_strategy']}

Suggested Repair Pattern:
{plan['repair_pattern']}

CVE Description:
{plan['cve_description']}

Fix the vulnerable C code below using the suggested repair strategy.
Keep the patch minimal and do not change unrelated logic.
Return only the fixed C code inside one code block.

Vulnerable Code:
{row['vulnerable_code']}
"""
    
    raw_output, patch_code = generate_patch(prompt)
    
    return raw_output, patch_code

In [77]:
import re

def extract_code_block(text):
    if text is None:
        return ""
        
    code_blocks = re.findall(r"```c(.*?)```", text, re.DOTALL)
    
    if code_blocks:
        return code_blocks[0].strip()
    else:
        return text.strip()


def generate_patch(prompt, model="deepseek/deepseek-r1-distill-qwen-14b", temperature=0):
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": "You are a secure C programmer. Return only C code."},
            {"role": "user", "content": prompt}
        ]
    )
    
    raw_output = response.choices[0].message.content
    
    patch_code = extract_code_block(raw_output)
    
    return raw_output, patch_code

In [78]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [79]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

In [80]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("model loaded")



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded


In [109]:
import re
import torch

def extract_code_block(text):
    if text is None:
        return ""
    code_blocks = re.findall(r"```(?:c|cpp)?\s*(.*?)```", text, re.DOTALL)
    if code_blocks:
        return code_blocks[0].strip()
    return text.strip()

def generate_patch(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,   # keep small
            do_sample=False,
            temperature=0.0,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only code block
    import re
    match = re.search(r"```(?:c|cpp)?\n(.*?)```", text, re.DOTALL)
    patch_code = match.group(1).strip() if match else text.strip()

    return text, patch_code

In [82]:
test_prompt = """
Fix the vulnerable C code below.
Return only the patched C code inside one code block.

```c
if (ptr) {
    len = strlen(ptr);
}
"""

raw, patch = generate_patch(test_prompt)

print("===== RAW OUTPUT =====")
print(raw[:800])

print("\n===== PATCH =====")
print(patch[:800])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


===== RAW OUTPUT =====
```c
if (ptr != NULL) {
    len = strlen(ptr);
}
```

===== PATCH =====
if (ptr != NULL) {
    len = strlen(ptr);
}


In [83]:
row = df_sample.iloc[0]

plan = planner_agent(row)
raw, patch = generator_agent(row, plan)

print("===== PLAN =====")
print(plan)

print("\n===== RAW OUTPUT =====")
print(raw[:1000])

print("\n===== PATCH =====")
print(patch[:1000])

===== PLAN =====
{'cwe_id': 'CWE-125', 'cve_description': "[{'lang': 'en', 'value': 'soundlib/Snd_fx.cpp in OpenMPT before 1.27.07.00 and libopenmpt before 0.3.8 allows remote attackers to cause a denial of service (out-of-bounds read) via an IT or MO3 file with many nested pattern loops.'}]", 'vulnerability_type': 'Out-of-bounds read', 'repair_strategy': 'Add bounds checking before reading from buffer.', 'repair_pattern': 'if (index < size) { ... }', 'risk': 'Memory disclosure'}

===== RAW OUTPUT =====
```
for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)
```

===== PATCH =====
for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)


In [103]:
# ---------------------------
# Agent 3: Validator Agent
# ---------------------------

def cwe_rule_check(cwe_id, patch):
    if patch is None:
        return False

    patch = patch.lower()

    if cwe_id == "CWE-125":  # Out-of-bounds read
        patterns = ["<", ">", "<=", ">=", "if", "&&", "||"]
        return any(p in patch for p in patterns)

    elif cwe_id == "CWE-787":  # Out-of-bounds write
        patterns = ["<", ">", "<=", ">=", "if", "&&"]
        return any(p in patch for p in patterns)

    elif cwe_id == "CWE-476":  # NULL pointer dereference
        patterns = ["if", "!=", "== null", "!= null"]
        return any(p in patch for p in patterns)

    elif cwe_id == "CWE-190":  # Integer overflow
        patterns = ["if", ">", "<", "limit", "max", "overflow"]
        return any(p in patch for p in patterns)

    elif cwe_id == "CWE-416":  # Use-after-free
        patterns = ["= null", "free", "if"]
        return any(p in patch for p in patterns)

    else:
        return False


def validator_agent(row, patch_code):
    compile_ok = check_c_syntax(patch_code)
    sim = similarity_score(patch_code, row["fixed_code"])
    codebleu = get_codebleu_score(patch_code, row["fixed_code"])
    cwe_fix = cwe_rule_check(row["cwe_id"], patch_code)

    return {
        "compile_ok": compile_ok,
        "similarity": sim,
        "codebleu": codebleu,
        "cwe_fix": cwe_fix
    }

In [85]:
import subprocess
import tempfile
import os
from difflib import SequenceMatcher

# Check if C code compiles
def check_c_syntax(code):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".c") as f:
            f.write(code.encode())
            fname = f.name

        result = subprocess.run(
            ["gcc", "-fsyntax-only", fname],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )

        os.remove(fname)
        return 1 if result.returncode == 0 else 0
    except:
        return 0


# Similarity score (paper metric)
def similarity_score(pred, ref):
    return SequenceMatcher(None, str(pred), str(ref)).ratio()


# Simple CodeBLEU approximation (token overlap)
def get_codebleu_score(pred, ref):
    pred_tokens = set(str(pred).split())
    ref_tokens = set(str(ref).split())

    if len(ref_tokens) == 0:
        return 0

    overlap = len(pred_tokens & ref_tokens)
    return overlap / len(ref_tokens)

In [86]:
val = validator_agent(row, patch)

print("===== PATCH =====")
print(patch)

print("\n===== VALIDATION =====")
print(val)

===== PATCH =====
for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)

===== VALIDATION =====
{'compile_ok': 0, 'similarity': 0.659217877094972, 'codebleu': 0.5, 'cwe_fix': False}


In [87]:
# ---------------------------
# Agent 4: Critic Agent
# ---------------------------

def critic_agent(row, patch_code, val_result, plan):
    reasons = []

    if val_result["compile_ok"] == 0:
        reasons.append("The patch is syntactically incomplete or does not form valid C code.")

    if val_result["cwe_fix"] is False:
        reasons.append(
            f"The patch does not apply the required repair strategy for {plan['cwe_id']} ({plan['vulnerability_type']})."
        )
        reasons.append(f"Required strategy: {plan['repair_strategy']}")
        reasons.append(f"Expected repair pattern: {plan['repair_pattern']}")

    if val_result["similarity"] < 0.3:
        reasons.append("The patch is too far from the reference fix and likely changed the code incorrectly.")

    if val_result["codebleu"] < 0.2:
        reasons.append("The patch has poor structural overlap with the expected fix.")

    if not reasons:
        reasons.append("The patch can still be improved while keeping the change minimal and vulnerability-focused.")

    critique = " ".join(reasons)
    return critique

In [88]:
critique = critic_agent(row, patch, val, plan)

print("===== CRITIQUE =====")
print(critique)

===== CRITIQUE =====
The patch is syntactically incomplete or does not form valid C code. The patch does not apply the required repair strategy for CWE-125 (Out-of-bounds read). Required strategy: Add bounds checking before reading from buffer. Expected repair pattern: if (index < size) { ... }


In [89]:
def refiner_agent(row, plan, previous_patch, critique):
    prompt = f"""
You are a secure C programmer.
Vulnerability Type: {plan['vulnerability_type']}
CWE ID: {plan['cwe_id']}
Risk: {plan['risk']}
Repair Strategy:
{plan['repair_strategy']}
Expected Repair Pattern:
{plan['repair_pattern']}
Previous patch:
```c
{previous_patch}
```
Critique:
{critique}
Now fix the vulnerable C code properly.
Vulnerable Code:
{row['vulnerable_code']}
Return only patched C code inside one code block.
"""
    raw_output, refined_patch = generate_patch(prompt)
    return raw_output, refined_patch

In [90]:
# ---------------------------
# Agent 5: Refiner Agent
# ---------------------------
def refiner_agent(row, plan, previous_patch, critique):
    prompt = f"""
You are a secure C programmer.
Vulnerability Type: {plan['vulnerability_type']}
CWE ID: {plan['cwe_id']}
Risk: {plan['risk']}
Repair Strategy:
{plan['repair_strategy']}
Expected Repair Pattern:
{plan['repair_pattern']}
Previous patch:
```c
{previous_patch}
```
Critique:
{critique}
Now fix the vulnerable C code properly.
Vulnerable Code:
{row['vulnerable_code']}
Return only patched C code inside one code block.
"""
    raw_output, refined_patch = generate_patch(prompt)
    return raw_output, refined_patch

In [91]:
raw2, refined_patch = refiner_agent(row, plan, patch, critique)
val2 = validator_agent(row, refined_patch)

print("===== REFINED PATCH =====")
print(refined_patch)

print("\n===== REFINED VALIDATION =====")
print(val2)

===== REFINED PATCH =====
#include <stdio.h>

// Assume GetNumChannels() and MAX_CHANNELS are defined elsewhere in your code
int GetNumChannels() {
    // Implementation of GetNumChannels function
}

#define MAX_CHANNELS 100

void safeRead(int index, int* value) {
    if (index >= 0 && index < MAX_CHANNELS) {
        *value = GetNumChannels();
    } else {
        printf("Index out of bounds\n");
        *value = -1; // Or any default value you prefer
    }
}

int main() {
    CHANNELINDEX nChn = 0;
    int result;

    safeRead(nChn, &result);

    printf("Result: %d\n", result);

    return 0;
}

===== REFINED VALIDATION =====
{'compile_ok': 0, 'similarity': 0.14222222222222222, 'codebleu': 0.5, 'cwe_fix': True}


In [92]:
def refiner_agent(row, plan, previous_patch, critique):
    prompt = f"""You are a secure C vulnerability repair system.
Task: Repair the vulnerable code.
Rules:
1. Return only the patched snippet.
2. Do not add includes.
3. Do not add main function.
4. Do not add helper functions.
5. Do not add type definitions.
6. Do not rewrite the whole program.
7. Keep the patch minimal.
8. Preserve the original structure.
9. Apply the required repair strategy.
Vulnerability Type: {plan['vulnerability_type']}
CWE ID: {plan['cwe_id']}
Risk: {plan['risk']}
Repair Strategy:
{plan['repair_strategy']}
Expected Repair Pattern:
{plan['repair_pattern']}
Previous patch:
{previous_patch}
Critique: {critique}
Now fix the vulnerable C code properly.
Vulnerable Code:
{row['vulnerable_code']}
Return only patched C code inside one code block."""

    raw_output, refined_patch = generate_patch(prompt)
    return raw_output, refined_patch

In [93]:
raw2, refined_patch = refiner_agent(row, plan, patch, critique)
val2 = validator_agent(row, refined_patch)

print("===== REFINED PATCH =====")
print(refined_patch)

print("\n===== REFINED VALIDATION =====")
print(val2)

===== REFINED PATCH =====
if (index < GetNumChannels()) {
    // Safe access to GetNumChannels()
    CHANNELINDEX numChannels = GetNumChannels();
    // Use numChannels safely within the loop
    for(CHANNELINDEX nChn = 0; nChn < numChannels; nChn++) {
        // Your code here
    }
}

===== REFINED VALIDATION =====
{'compile_ok': 0, 'similarity': 0.3641456582633053, 'codebleu': 0.6, 'cwe_fix': True}


In [94]:
def agentic_repair(row, max_iters=3):
    history = []

    def normalize_compile_result(comp):
        if isinstance(comp, dict):
            return {
                "ok": comp.get("ok", False),
                "stderr": comp.get("stderr", "")
            }
        elif isinstance(comp, tuple):
            # expected like (ok, stderr) or similar
            ok = comp[0] if len(comp) > 0 else False
            stderr = comp[1] if len(comp) > 1 else ""
            return {"ok": ok, "stderr": stderr}
        else:
            return {"ok": False, "stderr": str(comp)}

    def normalize_cppcheck_result(static):
        if isinstance(static, dict):
            return {
                "warning_count": static.get("warning_count", 0),
                "warnings": static.get("warnings", "")
            }
        elif isinstance(static, tuple):
            wc = static[0] if len(static) > 0 else 0
            ws = static[1] if len(static) > 1 else ""
            return {"warning_count": wc, "warnings": ws}
        else:
            return {"warning_count": 0, "warnings": str(static)}

    prompt = build_patch_prompt(row)
    raw_output, patch = generate_patch(prompt)

    for i in range(max_iters):
        comp = normalize_compile_result(compile_check(patch, row.get("language", "c")))
        static = normalize_cppcheck_result(cppcheck_check(patch, row.get("language", "c")))
        unsafe = unsafe_api_hits(patch)

        history.append({
            "iter": i,
            "compile_ok": comp["ok"],
            "warnings": static["warning_count"],
            "unsafe_hits": unsafe,
            "patch": patch[:500]
        })

        if comp["ok"] and static["warning_count"] == 0 and len(unsafe) == 0:
            break

        critique = f"""
The patch has issues:
Compile OK: {comp['ok']}
Compiler errors: {comp['stderr']}
Warnings: {static['warning_count']}
Cppcheck output: {static['warnings']}
Unsafe APIs: {unsafe}

Improve the patch.
Keep changes minimal.
Return only the full patched code inside one code block.
"""

        refine_prompt = build_patch_prompt(row) + "\n\n" + critique
        raw_output, patch = generate_patch(refine_prompt)

    return patch, history

In [95]:
row = df_sample.iloc[0]

final_patch, hist = agentic_repair(row)

print("===== FINAL PATCH =====")
print(final_patch[:800])

print("\n===== HISTORY =====")
for h in hist:
    print(h)

===== FINAL PATCH =====
#include <stdio.h>

#define CHANNELINDEX int

int GetNumChannels() {
    // Assume this function returns the number of channels
    return 10;
}

int main() {
    for (CHANNELINDEX nChn = 0; nChn < GetNumChannels(); nChn++) {
        // Safe read operation
        printf("Channel %d\n", nChn);
    }
    return 0;
}

===== HISTORY =====
{'iter': 0, 'compile_ok': False, 'warnings': 3, 'unsafe_hits': [], 'patch': '```c\n\t\t\t\tfor(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)'}
{'iter': 1, 'compile_ok': False, 'warnings': 0, 'unsafe_hits': [], 'patch': '#include <stdio.h>\n\n#define MAX_CHANNELS 100\n\nvoid processChannels(int numChannels) {\n    for (int nChn = 0; nChn < numChannels && nChn < MAX_CHANNELS; nChn++) {\n        // Process each channel\n        printf("Processing channel %d\\n", nChn);\n    }\n}\n\nint main() {\n    int numChannels = GetNumChannels();\n    processChannels(numChannels);\n    return 0;\n}'}
{'iter': 2, 

In [96]:

def agentic_repair(row, max_iters=3):
    history = []

    def normalize_compile_result(comp):
        if isinstance(comp, dict):
            return {"ok": comp.get("ok", False), "stderr": comp.get("stderr", "")}
        elif isinstance(comp, tuple):
            ok = comp[0] if len(comp) > 0 else False
            stderr = comp[1] if len(comp) > 1 else ""
            return {"ok": ok, "stderr": stderr}
        else:
            return {"ok": False, "stderr": str(comp)}

    def normalize_cppcheck_result(static):
        if isinstance(static, dict):
            return {"warning_count": static.get("warning_count", 0), "warnings": static.get("warnings", "")}
        elif isinstance(static, tuple):
            wc = static[0] if len(static) > 0 else 0
            ws = static[1] if len(static) > 1 else ""
            return {"warning_count": wc, "warnings": ws}
        else:
            return {"warning_count": 0, "warnings": str(static)}

    prompt = build_patch_prompt_strict(row)
    raw_output, patch = generate_patch(prompt)

    for i in range(max_iters):
        comp = normalize_compile_result(compile_check(patch, row.get("language", "c")))
        static = normalize_cppcheck_result(cppcheck_check(patch, row.get("language", "c")))
        unsafe = unsafe_api_hits(patch)

        history.append({
            "iter": i,
            "compile_ok": comp["ok"],
            "warnings": static["warning_count"],
            "unsafe_hits": unsafe,
            "patch": patch[:500]
        })

        if comp["ok"] and static["warning_count"] == 0 and len(unsafe) == 0:
            break

        critique = f"""
The previous patch is not acceptable.

Problems:
- Compile OK: {comp['ok']}
- Compiler errors: {comp['stderr']}
- Warnings: {static['warning_count']}
- Unsafe APIs: {unsafe}

IMPORTANT:
- Keep the original snippet structure.
- Do not add includes.
- Do not add main().
- Do not add helper functions.
- Only modify the vulnerable lines.
- Return only the patched snippet inside one code block.
"""

        refine_prompt = build_patch_prompt_strict(row) + "\n\n" + critique
        raw_output, patch = generate_patch(refine_prompt)

    return patch, history

In [107]:
def build_patch_prompt_strict(row):
    before = str(row.get("vulnerable_code", ""))
    cwe = str(row.get("cwe_id", ""))
    cve = str(row.get("cve_id", ""))

    # truncate long code to avoid OOM
    max_chars = 1200
    if len(before) > max_chars:
        before = before[:max_chars]

    strategy = CWE_STRATEGY.get(cwe, "Fix the vulnerability safely with minimal changes.")

    return f"""
You are a secure C/C++ patch generator.

CVE: {cve}
CWE: {cwe}
Repair strategy: {strategy}

STRICT RULES:
1. Preserve the original code structure.
2. Do NOT add #include lines.
3. Do NOT add main().
4. Do NOT add helper functions.
5. Do NOT add dummy implementations.
6. Do NOT rewrite the whole file.
7. Only patch the given snippet.
8. Keep the patch minimal.
9. Return ONLY one fenced code block.

VULNERABLE CODE:
```c
{before}
""".strip()

In [108]:
row = df_sample.iloc[0]
p = build_patch_prompt_strict(row)
print(len(p))
print(p[:700])

589
You are a secure C/C++ patch generator.

CVE: CVE-2018-10017
CWE: CWE-125
Repair strategy: Focus on out-of-bounds READ prevention: bounds checks before reads, validate indices, validate length.

STRICT RULES:
1. Preserve the original code structure.
2. Do NOT add #include lines.
3. Do NOT add main().
4. Do NOT add helper functions.
5. Do NOT add dummy implementations.
6. Do NOT rewrite the whole file.
7. Only patch the given snippet.
8. Keep the patch minimal.
9. Return ONLY one fenced code block.

VULNERABLE CODE:
```c
				for(CHANNELINDEX nChn = 0; nChn < GetNumChannels(); nChn++)


In [99]:
print(wrap_if_needed("for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)"))

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>

for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)

int main(){return 0;}



In [100]:
row = df_sample.iloc[0]
final_patch, hist = agentic_repair(row)

print("===== FINAL PATCH =====")
print(final_patch)

print("\n===== HISTORY =====")
for h in hist:
    print(h)

===== FINAL PATCH =====
for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)

===== HISTORY =====
{'iter': 0, 'compile_ok': False, 'warnings': 3, 'unsafe_hits': [], 'patch': '{\n    if (nChn >= GetNumChannels()) {\n        // Handle out-of-bounds read\n        return;\n    }\n\n    // Safe read operation\n    int value = GetChannelValue(nChn);\n}'}
{'iter': 1, 'compile_ok': False, 'warnings': 3, 'unsafe_hits': [], 'patch': 'for (CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)'}
{'iter': 2, 'compile_ok': False, 'warnings': 3, 'unsafe_hits': [], 'patch': 'for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)'}


In [101]:
def agentic_repair(row, max_iters=3):
    history = []
    best_patch = None
    best_score = -1

    def normalize_compile_result(comp):
        if isinstance(comp, dict):
            return {"ok": comp.get("ok", False), "stderr": comp.get("stderr", "")}
        elif isinstance(comp, tuple):
            ok = comp[0] if len(comp) > 0 else False
            stderr = comp[1] if len(comp) > 1 else ""
            return {"ok": ok, "stderr": stderr}
        else:
            return {"ok": False, "stderr": str(comp)}

    prompt = build_patch_prompt_strict(row)
    raw_output, patch = generate_patch(prompt)

    for i in range(max_iters):
        comp = normalize_compile_result(compile_check(patch, row.get("language", "c")))
        sim = similarity_score(patch, row["fixed_code"])
        cwe_fix = cwe_rule_check(row["cwe_id"], patch)

        score = (2 * int(comp["ok"])) + (2 * int(cwe_fix)) + sim

        history.append({
            "iter": i,
            "compile_ok": comp["ok"],
            "cwe_fix": cwe_fix,
            "similarity": sim,
            "score": score,
            "patch": patch[:500]
        })

        if score > best_score:
            best_score = score
            best_patch = patch

        if comp["ok"] and cwe_fix:
            break

        critique = f"""
The previous patch is not acceptable.

Problems:
- Compile OK: {comp['ok']}
- Compiler errors: {comp['stderr']}
- CWE fix satisfied: {cwe_fix}
- Similarity: {sim}

IMPORTANT:
- Keep the original snippet structure.
- Do not add includes.
- Do not add main().
- Do not add helper functions.
- Only modify the vulnerable lines.
- Return only the patched snippet inside one code block.
"""

        refine_prompt = build_patch_prompt_strict(row) + "\n\n" + critique
        raw_output, patch = generate_patch(refine_prompt)

    return best_patch, history

In [104]:
row = df_sample.iloc[0]
final_patch, hist = agentic_repair(row)

print("===== FINAL PATCH =====")
print(final_patch)

print("\n===== HISTORY =====")
for h in hist:
    print(h)

===== FINAL PATCH =====
for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)

===== HISTORY =====
{'iter': 0, 'compile_ok': False, 'cwe_fix': True, 'similarity': 0.2366412213740458, 'score': 2.236641221374046, 'patch': '{\n    if (nChn >= GetNumChannels()) {\n        // Handle out-of-bounds read\n        return;\n    }\n\n    // Safe read operation\n    int value = GetChannelValue(nChn);\n}'}
{'iter': 1, 'compile_ok': False, 'cwe_fix': True, 'similarity': 0.6555555555555556, 'score': 2.6555555555555554, 'patch': 'for (CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)'}
{'iter': 2, 'compile_ok': False, 'cwe_fix': True, 'similarity': 0.659217877094972, 'score': 2.659217877094972, 'patch': 'for(CHANNELINDEX nChn = 0; nChn < GetNumChannels() && nChn < MAX_CHANNELS; nChn++)'}


In [110]:
results_agentic = []

for i in range(10):
    row = df_sample.iloc[i]
    
    final_patch, hist = agentic_repair(row)
    
    comp = compile_check(final_patch, row.get("language","c"))
    sim = similarity_score(final_patch, row["fixed_code"])
    cwe_fix = cwe_rule_check(row["cwe_id"], final_patch)
    codebleu = get_codebleu_score(final_patch, row["fixed_code"])
    
    results_agentic.append({
        "cve_id": row["cve_id"],
        "cwe_id": row["cwe_id"],
        "compile_ok": comp[0] if isinstance(comp, tuple) else comp["ok"],
        "cwe_fix": cwe_fix,
        "similarity": sim,
        "codebleu": codebleu
    })

import pandas as pd
results_agentic_df = pd.DataFrame(results_agentic)
results_agentic_df

,cve_id,cwe_id,compile_ok,cwe_fix,similarity,codebleu
0,CVE-2018-10017,CWE-125,False,True,0.450000,0.600000
1,CVE-2018-16749,CWE-476,False,True,0.132196,0.333333
2,CVE-2021-22552,CWE-125,True,True,0.121212,0.181818
3,CVE-2019-19274,CWE-125,False,True,0.019289,0.041839
4,CVE-2022-0712,CWE-476,True,False,0.464567,0.692308
5,CVE-2020-1921,CWE-787,False,True,0.009562,0.240000
6,CVE-2017-6800,CWE-125,False,True,0.736842,0.500000
7,CVE-2018-20455,CWE-787,False,True,0.664557,0.500000
8,CVE-2016-9084,CWE-190,False,True,0.521505,0.680000
9,CVE-2022-3213,CWE-787,False,True,0.680000,0.000000


In [116]:
print("Agentic Compile Rate:", results_agentic_df["compile_ok"].mean())
print("Agentic CWE Fix Rate:", results_agentic_df["cwe_fix"].mean())
print("Agentic Mean Similarity:", results_agentic_df["similarity"].mean())
print("Agentic Mean CodeBLEU:", results_agentic_df["codebleu"].mean())

Agentic Compile Rate: 0.2
Agentic CWE Fix Rate: 0.9
Agentic Mean Similarity: 0.37997303832830676
Agentic Mean CodeBLEU: 0.3769298050434414
